In [15]:
import pandas as pd

## Metadata of patches
Our pipeline accepts svs at a time so to avoid overwritting in the process_list_autogen.csv, we are creating our own list

In [1]:
# import pandas as pd

# # Hardcoded column names
# columns = [
#     'slide_id','seg_level', 'sthresh', 'mthresh', 'close', 'use_otsu',
#     'keep_ids', 'exclude_ids', 'a_t', 'a_h', 'max_n_holes',
#     'use_padding', 'contour_fn', 'vis_level', 'line_thickness'
# ]

# # Create empty DataFrame with only headers
# df = pd.DataFrame(columns=columns)

# # Save to CSV
# df.to_csv("./FLAT_DIRECTORY/process_list_final.csv", index=False)

In [3]:
# import pandas as pd

# # Hardcoded column names
# columns = [
#     'svs_file','renamed_file'
# ]

# # Create empty DataFrame with only headers
# df = pd.DataFrame(columns=columns)

# # Save to CSV
# df.to_csv("./FLAT_DIRECTORY/file_names.csv", index=False)

In [43]:
def is_valid_svs_file(filename, excel_path="PatientsSheet.xlsx", columnNumb=0):
    """
    Check if a given .svs filename is valid based on:
    - Matching first 12 characters with patient ID in Excel column 0.
    - Patient's label in specified column is not NaN or empty.

    Args:
        filename (str): Filename to check (should end in .svs).
        excel_path (str): Path to the Excel sheet.
        columnNumb (int): Column number to check for the label (default: 1 for MSI).

    Returns:
        tuple: (bool, value) where bool is True if valid, False otherwise,
               and value is the label from the specified column or None if invalid.
    """
    # Check if filename ends with .svs
    if not filename.lower().endswith('.svs'):
        print(f"Error: Filename '{filename}' does not end with .svs")
        return False, None

    # Check if filename is long enough to extract patient ID
    if len(filename) < 12:
        print(f"Error: Filename '{filename}' is too short (less than 12 characters)")
        return False, None

    # Extract patient ID from first 12 characters
    patient_id = filename[:12]

    try:
        # Read Excel file
        df = pd.read_excel(excel_path, header=None)
    except FileNotFoundError:
        print(f"Error: Excel file '{excel_path}' not found.")
        return False, None
    except Exception as e:
        print(f"Error reading Excel file: {e}")
        return False, None

    # Check if DataFrame is empty
    if df.empty:
        print("Error: Excel file is empty.")
        return False, None

    # Handle headers - skip first row if it contains "PATIENTS"
    if len(df) > 0 and str(df.iloc[0, 0]).strip().upper() == "PATIENTS":
        df = df.iloc[1:]

    # Check if we have data after potential header removal
    if df.empty:
        print("Error: No data found after header processing.")
        return False, None

    # Ensure we have enough columns (at least columnNumb + 1 since it's 0-indexed)
    required_columns = max(1, columnNumb + 1)  # At least 1 column, or columnNumb + 1
    if df.shape[1] < required_columns:
        print(f"Error: Excel file must have at least {required_columns} columns. "
              f"Found {df.shape[1]} columns. Cannot access column {columnNumb}.")
        return False, None

    # Clean and standardize patient ID column (always column 0)
    df.iloc[:, 0] = df.iloc[:, 0].astype(str).str.strip()
    
    # Clean and standardize the target column
    df.iloc[:, columnNumb] = df.iloc[:, columnNumb].astype(str).str.strip()

    # Filter for this patient
    match = df[df.iloc[:, 0] == patient_id]

    if match.empty:
        print(f"Patient ID '{patient_id}' not found in Excel file.")
        return False, None

    # Check label validity in specified column
    label = match.iloc[0, columnNumb]
    if pd.isna(label) or str(label).strip().lower() in ['nan', '', 'none']:
        print(f"Patient '{patient_id}' has invalid or empty label in column {columnNumb}: '{label}'")
        return False, None

    print(f"Valid file: Patient '{patient_id}' found with label in column {columnNumb}: '{label}'")
    return True, label

## Renaming files to patientid and label
Clam on default give name of svs to .h5 file so we are renaming our .h5 file to 12 char of .svs file with label

In [16]:
def rename_h5_file(svs_filename, label, h5_dir="./FLAT_DIRECTORY/patches"):
    """
    Rename .h5 file to {first_12_chars}_{label}.h5
    """
    base_name, _ = os.path.splitext(svs_filename)
    new_base = f"{svs_filename[:12]}_{label}.h5"

    # Find the matching .h5 file created
    old_h5_path = os.path.join(h5_dir, f"{base_name}.h5")
    new_h5_path = os.path.join(h5_dir, new_base)

    if os.path.exists(old_h5_path):
        os.rename(old_h5_path, new_h5_path)
        print(f"📦 Renamed H5: {old_h5_path} → {new_h5_path}")
    else:
        print(f"⚠️ H5 file not found: {old_h5_path}")

    return new_h5_path

## Patch creation function

In [45]:
import os
import h5py
import openslide
from PIL import Image

def create_patches_from_h5(h5_path, svs_path, output_dir, magnification):
    os.makedirs(output_dir, exist_ok=True)

    # Define patch size based on magnification
    if magnification == 40:
        PATCH_SIZE = 1024
        FINAL_SIZE = 512  # Downsample to 512
    elif magnification == 20:
        PATCH_SIZE = 512
        FINAL_SIZE = 512  # No downsampling needed
    else:
        raise ValueError(f"Unsupported magnification: {magnification}")

    # Extract base name from h5 file (without extension)
    base_name = os.path.splitext(os.path.basename(h5_path))[0]
    
    # Load coordinates
    with h5py.File(h5_path, 'r') as h5_file:
        coords = h5_file['coords'][:]  # shape (N, 2), each row is (x, y)
    
    # Open the WSI
    slide = openslide.OpenSlide(svs_path)
    
    # Extract and save patches
    for i, (x, y) in enumerate(coords):
        patch = slide.read_region((int(x), int(y)), 0, (PATCH_SIZE, PATCH_SIZE)).convert('RGB')

        # Downsample if 40x (1024 -> 512)
        if PATCH_SIZE != FINAL_SIZE:
            patch = patch.resize((FINAL_SIZE, FINAL_SIZE), resample=Image.BICUBIC)
        
        # Save using the original x, y coordinates
        patch_name = f"{base_name}_x{int(x)}_y{int(y)}_patch{i:05d}.png"
        patch.save(os.path.join(output_dir, patch_name))
    
    print(f"✅ Done! Saved {len(coords)} patches to {output_dir}")


## Comparing to the slidelog metadata file

In [18]:
import os
import pandas as pd

def check_svs_file(svs_file, excel_file, filename_column="Filename", label_column="Label"):
    try:
        df = pd.read_csv(excel_file)

        filename = svs_file
        row = df[df[filename_column] == filename]

        if not row.empty:
            label = row.iloc[0][label_column]
            return True, label
        else:
            return False, None

    except Exception as e:
        print(f"Error reading Excel or processing file: {e}")
        return False, None


## Pipeline

In [52]:
import os
import subprocess
from pathlib import Path

# Top-level directory containing subfolders with .svs files
wsi_root = "D:/Aamir Gulzar/WSI_Raw_data"
flat_dir = "./FLAT_DIRECTORY"
count=0

# Loop through all subfolders
for subdir in os.listdir(wsi_root):
    #***************Checking for svs file***************#
    if count==2:
        break
    count+=1
    
    full_subdir_path = os.path.join(wsi_root, subdir) #D:/Aamir Gulzar/WSI_Raw_data/0ecb2d7b-0a69-46ce-80d6-1be65cec3cc7
    if not os.path.isdir(full_subdir_path):
        continue  # skip non-folder items

    # Search for .svs file in this subfolder
    svs_files = [f for f in os.listdir(full_subdir_path) if f.endswith('.svs')]
    if not svs_files:
        print(f"❌ No .svs file found in: {full_subdir_path}")
        continue
    else:
        print(svs_files)

    # If multiple .svs files exist, process each one
    for svs_file in svs_files:
        
        #***************Checking is svs is valid or not***************#
        bool_valid, label= check_svs_file(svs_file, "./slides_metadata/slide_metadata_filtered.csv", filename_column="Filename", label_column="Label")

        newnameofsvs= f"{svs_file[:12]}_{label}"
        if bool_valid==False:
            continue

        # print(bool_valid, newnameofsvs)

        #***************setting directories for wsi_raw_data***************#
        source_dir = full_subdir_path #D:/Aamir Gulzar/WSI_Raw_data/0ecb2d7b-0a69-46ce-80d6-1be65cec3cc7
        source_dir_quoted = f"{source_dir}" #D:/Aamir Gulzar/WSI_Raw_data/0ecb2d7b-0a69-46ce-80d6-1be65cec3cc7
        
        flat_dir_quoted = f"{flat_dir}" #./FLAT_DIRECTORY

        #***************setting directories for downsampled***************#
        output_dir= "./FLAT_DIRECTORY/downsampled"
        
        svs_path = os.path.join(source_dir_quoted, svs_file) #D:/Aamir Gulzar/WSI_Raw_data/0ecb2d7b-0a69-46ce-80d6-1be65cec3cc7/TCGA-AA-3524-01Z-00-DX1.b1aae264-87be-4514-8f9d-25660b39caa7.svs
        print(f"🚀 Processing: {svs_file} in {subdir}")
        
        slide = openslide.OpenSlide(svs_path)
        
        # Get objective power (magnification) from slide metadata
        mag = int(slide.properties.get("openslide.objective-power", 20))
        
        # Choose patch size and step size based on magnification
        if mag == 40:
            print("Magnification is 40x")
            patch_size = '1024'
            step_size = '1024'
        else:
            print("Magnification is 20x")
            patch_size = '512'
            step_size = '512'

        #***************Creating .h5 command***************#
        command = [
            'python', './CLAM/create_patches_fp.py',
            '--source', source_dir_quoted,
            '--save_dir', flat_dir,
            '--patch_size', patch_size,
            '--step_size', step_size,
            '--seg', '--patch'
        ]
        
        # Execute the command
                
        # # Use Popen to stream output line-by-line
        with subprocess.Popen(command, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True) as proc:
            for line in proc.stdout:
                print(line, end='')  # stream each line live

        #***************Renaming .h5 file***************#
        new_h5_path= rename_h5_file(svs_file, label, h5_dir="./FLAT_DIRECTORY/patches")

        # #***************Setting path for png patches***************#
        base_dir = "./FLAT_DIRECTORY"
        sub_dir = "patches"

        # Create full path
        print(new_h5_path)
        base_dir= "./Patches"
        sub_dir= f"{svs_file[:12]}_{label}"
        output_dir = os.path.join(base_dir, sub_dir)
        from pathlib import Path
        svs_base_filename = os.path.splitext(os.path.basename(svs_file))[0]
        filename = f"{svs_base_filename }skipped_patches.csv"  # Or .csv/.json/etc.
        full_path = Path("./FLAT_DIRECTORY/patches/") / filename
        
        #creating patches from h5 files
        # create_patches_from_h5(new_h5_path, svs_path, output_dir,full_path, mag)
        create_patches_from_h5(new_h5_path, svs_path, output_dir, mag)

['TCGA-A6-2686-01Z-00-DX1.0540a027-2a0c-46c7-9af0-7b8672631de7.svs']
🚀 Processing: TCGA-A6-2686-01Z-00-DX1.0540a027-2a0c-46c7-9af0-7b8672631de7.svs in 001b7d97-9425-43c3-a9a3-a36cb3d2a591
source:  D:/Aamir Gulzar/WSI_Raw_data\001b7d97-9425-43c3-a9a3-a36cb3d2a591
patch_save_dir:  ./FLAT_DIRECTORY\patches
mask_save_dir:  ./FLAT_DIRECTORY\masks
stitch_save_dir:  ./FLAT_DIRECTORY\stitches
source : D:/Aamir Gulzar/WSI_Raw_data\001b7d97-9425-43c3-a9a3-a36cb3d2a591
save_dir : ./FLAT_DIRECTORY
patch_save_dir : ./FLAT_DIRECTORY\patches
mask_save_dir : ./FLAT_DIRECTORY\masks
stitch_save_dir : ./FLAT_DIRECTORY\stitches
{'seg_params': {'seg_level': -1, 'sthresh': 8, 'mthresh': 7, 'close': 4, 'use_otsu': False, 'keep_ids': 'none', 'exclude_ids': 'none'}, 'filter_params': {'a_t': 100, 'a_h': 16, 'max_n_holes': 8}, 'patch_params': {'use_padding': True, 'contour_fn': 'four_pt'}, 'vis_params': {'vis_level': -1, 'line_thickness': 250}}
Computed patch_level = 0 for 40x WSI (target 20x)

100%|##########| 

## Testing by reconstructing image

In [66]:
import os
import re
import numpy as np
import pandas as pd
from PIL import Image
from tqdm import tqdm

def reconstruct_from_patches(patch_dir, skipped_csv_path, output_path, patch_size=512):
    # Load skipped coordinates
    skipped_df = pd.read_csv(skipped_csv_path, usecols=['x', 'y'])

    # Parse all available patch filenames
    patch_files = [f for f in os.listdir(patch_dir) if f.endswith('.png')]
    coords_dict = {}

    # Regular expression to extract x, y from filename
    pattern = re.compile(r'_x(\d+)_y(\d+)_patch\d+\.png')

    for file in patch_files:
        match = pattern.search(file)
        if match:
            x, y = map(int, match.groups())
            coords_dict[(x, y)] = file

    # Combine available + skipped coordinates
    all_coords = set(coords_dict.keys()).union(set([tuple(row) for row in skipped_df[['x', 'y']].values]))

    # Determine full canvas size
    all_x = [x for x, _ in all_coords]
    all_y = [y for _, y in all_coords]
    min_x, max_x = min(all_x), max(all_x)
    min_y, max_y = min(all_y), max(all_y)

    width = (max_x - min_x + patch_size) // 2
    height = (max_y - min_y + patch_size) // 2


    n_cols = (width // patch_size)
    n_rows = (height // patch_size)

    print(f"🧱 Reconstructing image of size: {width} x {height} ({n_cols} cols x {n_rows} rows)")

    # Create blank white canvas
    canvas = Image.new('RGB', (width, height), color='white')

    for (x, y) in tqdm(all_coords, desc='🧩 Placing patches'):
        if (x, y) in coords_dict:
            patch_path = os.path.join(patch_dir, coords_dict[(x, y)])
            patch = Image.open(patch_path).convert('RGB')
        else:
            patch = Image.new('RGB', (patch_size, patch_size), color='white')

        scaled_x = (x - min_x) // 2
        scaled_y = (y - min_y) // 2
        canvas.paste(patch, (scaled_x, scaled_y))


    # Ensure output directory exists
    os.makedirs(os.path.dirname(output_path), exist_ok=True)

    # Save the reconstructed image
    canvas.save(output_path)
    print(f"✅ Reconstructed image saved to: {output_path}")

In [69]:
reconstruct_from_patches(
    patch_dir='./Patches/TCGA-A6-2686_MSIH',
    skipped_csv_path='./FLAT_DIRECTORY/patches/TCGA-A6-2686-01Z-00-DX1.0540a027-2a0c-46c7-9af0-7b8672631de7_skipped_patches.csv',
    output_path='./Reconstructed_img/TCGA-A6-2686.png'
)

🧱 Reconstructing image of size: 60442 x 28592 (118 cols x 55 rows)


🧩 Placing patches: 100%|█████████████████████████████████████████████████████████| 2260/2260 [00:17<00:00, 127.39it/s]


✅ Reconstructed image saved to: ./Reconstructed_img/TCGA-A6-2686.png


## Testing pipeline

In [18]:
import os
import openslide

wsi_root = "D:/Aamir Gulzar/WSI_Raw_data"
found_20x_path = None

for subdir in os.listdir(wsi_root):
    full_subdir_path = os.path.join(wsi_root, subdir)
    if not os.path.isdir(full_subdir_path):
        continue  # Skip if not a directory

    svs_files = [f for f in os.listdir(full_subdir_path) if f.endswith('.svs')]
    if not svs_files:
        print(f"❌ No .svs file found in: {full_subdir_path}")
        continue
    else:
        print(f"Found .svs files: {svs_files} in {subdir}")

    for svs_file in svs_files:
        svs_path = os.path.join(full_subdir_path, svs_file)
        try:
            slide = openslide.OpenSlide(svs_path)
            magnification = slide.properties.get("openslide.objective-power")
            slide.close()
            
            if magnification:
                print(f"File: {svs_file} | Magnification: {magnification}x")
                if magnification == "40":
                    found_20x_path = svs_path
                    print(f"✅ Found 20x magnification at: {found_20x_path}")
            else:
                print(f"File: {svs_file} | Magnification info not found")
        except Exception as e:
            print(f"⚠️ Error reading {svs_file}: {e}")

    # if found_20x_path:
    #     break  # Break outer loop once found

if not found_20x_path:
    print("No 20x magnification SVS file found.")
else:
    print(f"Stored path for 20x SVS file: {found_20x_path}")


Found .svs files: ['TCGA-A6-2686-01Z-00-DX1.0540a027-2a0c-46c7-9af0-7b8672631de7.svs'] in 001b7d97-9425-43c3-a9a3-a36cb3d2a591
File: TCGA-A6-2686-01Z-00-DX1.0540a027-2a0c-46c7-9af0-7b8672631de7.svs | Magnification: 40x
✅ Found 20x magnification at: D:/Aamir Gulzar/WSI_Raw_data\001b7d97-9425-43c3-a9a3-a36cb3d2a591\TCGA-A6-2686-01Z-00-DX1.0540a027-2a0c-46c7-9af0-7b8672631de7.svs
Found .svs files: ['TCGA-D5-5539-01Z-00-DX1.9c46fe78-2adb-4f49-9141-cda135c2c90b.svs'] in 00d69f69-3120-4b0c-8985-aab4edf9bc4b
File: TCGA-D5-5539-01Z-00-DX1.9c46fe78-2adb-4f49-9141-cda135c2c90b.svs | Magnification: 40x
✅ Found 20x magnification at: D:/Aamir Gulzar/WSI_Raw_data\00d69f69-3120-4b0c-8985-aab4edf9bc4b\TCGA-D5-5539-01Z-00-DX1.9c46fe78-2adb-4f49-9141-cda135c2c90b.svs
Found .svs files: ['TCGA-A6-2683-01Z-00-DX1.0dfc5d0a-68f4-45e1-a879-0428313c6dbc.svs'] in 011da0ba-03eb-41cd-ae90-2dca944410e5
File: TCGA-A6-2683-01Z-00-DX1.0dfc5d0a-68f4-45e1-a879-0428313c6dbc.svs | Magnification: 40x
✅ Found 20x magnifica

In [6]:
import openslide
svs_path= "D:/Aamir Gulzar/WSI_Raw_data/021821c9-68f3-4be3-97ec-4ade5b4aaa60/TCGA-AY-4070-01Z-00-DX1.dd650ac6-8480-4fd8-85b8-15a7840a5933.svs"
# "D:\Aamir Gulzar\WSI_Raw_data\021821c9-68f3-4be3-97ec-4ade5b4aaa60\TCGA-AY-4070-01Z-00-DX1.dd650ac6-8480-4fd8-85b8-15a7840a5933.svs"
slide = openslide.OpenSlide(svs_path)
magnification = slide.properties.get("openslide.objective-power")
slide.close()

if magnification:
    print(f"Magnification: {magnification}x")
    if magnification == "40":
        found_20x_path = svs_path
        print(f"✅ Found 40 magnification at: {found_20x_path}")
else:
    print(f"File: | Magnification info not found")

Magnification: 40x
✅ Found 40 magnification at: D:/Aamir Gulzar/WSI_Raw_data/021821c9-68f3-4be3-97ec-4ade5b4aaa60/TCGA-AY-4070-01Z-00-DX1.dd650ac6-8480-4fd8-85b8-15a7840a5933.svs


In [7]:
# Load the SVS file
slide = openslide.OpenSlide(svs_path)

# Get dimensions of the base (level 0)
width, height = slide.dimensions
print(f"Base level dimensions: {width} x {height}")

# Get number of levels
level_count = slide.level_count
print(f"Number of levels: {level_count}")

# Print dimensions for each level
print("Level dimensions:")
for i in range(level_count):
    print(f" Level {i}: {slide.level_dimensions[i]}")

# Optional: print additional metadata
print("\nProperties:")
for key, value in slide.properties.items():
    print(f"{key}: {value}")

Base level dimensions: 113191 x 21115
Number of levels: 4
Level dimensions:
 Level 0: (113191, 21115)
 Level 1: (28297, 5278)
 Level 2: (7074, 1319)
 Level 3: (3537, 659)

Properties:
aperio.AppMag: 40
aperio.DSR ID: aperio01
aperio.Date: 06/20/11
aperio.DisplayColor: 0
aperio.Exposure Scale: 0.000001
aperio.Exposure Time: 109
aperio.Filename: 26730
aperio.Focus Offset: 0.000000
aperio.ICC Profile: ScanScope v1
aperio.ImageID: 26730
aperio.Left: 24.250113
aperio.LineAreaXOffset: 0.000000
aperio.LineAreaYOffset: 0.000000
aperio.LineCameraSkew: -0.000389
aperio.MPP: 0.2520
aperio.OriginalHeight: 21215
aperio.OriginalWidth: 118000
aperio.Parmset: EPC
aperio.ScanScope ID: SS1302
aperio.StripeWidth: 1000
aperio.Time: 14:39:52
aperio.Time Zone: GMT-07:00
aperio.Top: 14.048120
aperio.User: bf4e4f93-d36f-4dd6-9328-5cbbf1164855
openslide.associated.thumbnail.height: 191
openslide.associated.thumbnail.width: 1024
openslide.comment: Aperio Image Library v10.2.41
118000x21215 [0,100 113191x21115] 

## Clam with 40x each of patch size 512

:/Aamir Gulzar/WSI_Raw_data/021821c9-68f3-4be3-97ec-4ade5b4aaa60/TCGA-AY-4070-01Z-00-DX1.dd650ac6-8480-4fd8-85b8-15a7840a5933.svs

In [10]:
import os
import subprocess
from pathlib import Path

# Path to the .svs file to test
# svs_path = found_20x_path
svs_filename = os.path.basename(svs_path)
source_dir = os.path.dirname(svs_path)

# Label check (you must have this function and metadata file available)
bool_valid, label = check_svs_file(svs_filename, "./slides_metadata/slide_metadata_filtered.csv", filename_column="Filename", label_column="Label")
# if not bool_valid:
#     print("❌ SVS file is not valid or not labeled properly.")
# else:
#     print(f"✅ SVS file is valid. Label: {label}")
    
newnameofsvs = f"{svs_filename[:12]}_{label}"
flat_dir = "./FLAT_DIRECTORY"

# Create patches using CLAM script
command = [
    'python', './CLAM/create_patches_fp.py',
    '--source', source_dir,
    '--save_dir', flat_dir,
    '--patch_size', '512',
    '--step_size', '512',
    '--seg', '--patch'
]

print(f"🚀 Running patch extraction for: {svs_filename}")
with subprocess.Popen(command, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True) as proc:
    for line in proc.stdout:
        print(line, end='')

# Rename .h5 file
h5_path = rename_h5_file(svs_filename, label, h5_dir="./FLAT_DIRECTORY/patches")
print(f"✅ Renamed h5 file: {h5_path}")

# Generate skipped patch filename
skipped_csv = Path("./FLAT_DIRECTORY/patches") / f"{os.path.splitext(svs_filename)[0]}_skipped_patches.csv"

# Create PNG patches from h5
png_patch_output_dir = os.path.join("./Patches", newnameofsvs)
print(f"🎨 Generating PNG patches in: {png_patch_output_dir}")
create_patches_from_h5(h5_path, svs_path, png_patch_output_dir)

🚀 Running patch extraction for: TCGA-AY-4070-01Z-00-DX1.dd650ac6-8480-4fd8-85b8-15a7840a5933.svs
source:  D:/Aamir Gulzar/WSI_Raw_data/021821c9-68f3-4be3-97ec-4ade5b4aaa60
patch_save_dir:  ./FLAT_DIRECTORY\patches
mask_save_dir:  ./FLAT_DIRECTORY\masks
stitch_save_dir:  ./FLAT_DIRECTORY\stitches
source : D:/Aamir Gulzar/WSI_Raw_data/021821c9-68f3-4be3-97ec-4ade5b4aaa60
save_dir : ./FLAT_DIRECTORY
patch_save_dir : ./FLAT_DIRECTORY\patches
mask_save_dir : ./FLAT_DIRECTORY\masks
stitch_save_dir : ./FLAT_DIRECTORY\stitches
{'seg_params': {'seg_level': -1, 'sthresh': 8, 'mthresh': 7, 'close': 4, 'use_otsu': False, 'keep_ids': 'none', 'exclude_ids': 'none'}, 'filter_params': {'a_t': 100, 'a_h': 16, 'max_n_holes': 8}, 'patch_params': {'use_padding': True, 'contour_fn': 'four_pt'}, 'vis_params': {'vis_level': -1, 'line_thickness': 250}}
Computed patch_level = 0 for 40x WSI (target 20x)

100%|##########| 1/1 [00:21<00:00, 21.64s/it]


progress: 0.00, 0/1
processing TCGA-AY-4070-01Z-00-DX1.dd650

## Clam with 10x each of patch size 512

:/Aamir Gulzar/WSI_Raw_data/021821c9-68f3-4be3-97ec-4ade5b4aaa60/TCGA-AY-4070-01Z-00-DX1.dd650ac6-8480-4fd8-85b8-15a7840a5933.svs

In [20]:
import os
import subprocess
from pathlib import Path

# Path to the .svs file to test
# svs_path = found_20x_path
svs_filename = os.path.basename(svs_path)
source_dir = os.path.dirname(svs_path)

# Label check (you must have this function and metadata file available)
bool_valid, label = check_svs_file(svs_filename, "./slides_metadata/slide_metadata_filtered.csv", filename_column="Filename", label_column="Label")
# if not bool_valid:
#     print("❌ SVS file is not valid or not labeled properly.")
# else:
#     print(f"✅ SVS file is valid. Label: {label}")
    
newnameofsvs = f"{svs_filename[:12]}_{label}"
flat_dir = "./FLAT_DIRECTORY"

# Create patches using CLAM script
command = [
    'python', './CLAM/create_patches_fp.py',
    '--source', source_dir,
    '--save_dir', flat_dir,
    '--patch_size', '512',
    '--step_size', '512',
    '--seg', '--patch'
]

print(f"🚀 Running patch extraction for: {svs_filename}")
with subprocess.Popen(command, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True) as proc:
    for line in proc.stdout:
        print(line, end='')

# Rename .h5 file
h5_path = rename_h5_file(svs_filename, label, h5_dir="./FLAT_DIRECTORY/patches")
print(f"✅ Renamed h5 file: {h5_path}")

# Generate skipped patch filename
skipped_csv = Path("./FLAT_DIRECTORY/patches") / f"{os.path.splitext(svs_filename)[0]}_skipped_patches.csv"

# Create PNG patches from h5
png_patch_output_dir = os.path.join("./Patches", newnameofsvs)
print(f"🎨 Generating PNG patches in: {png_patch_output_dir}")
create_patches_from_h5(h5_path, svs_path, png_patch_output_dir)

🚀 Running patch extraction for: TCGA-AY-4070-01Z-00-DX1.dd650ac6-8480-4fd8-85b8-15a7840a5933.svs
source:  D:/Aamir Gulzar/WSI_Raw_data/021821c9-68f3-4be3-97ec-4ade5b4aaa60
patch_save_dir:  ./FLAT_DIRECTORY\patches
mask_save_dir:  ./FLAT_DIRECTORY\masks
stitch_save_dir:  ./FLAT_DIRECTORY\stitches
source : D:/Aamir Gulzar/WSI_Raw_data/021821c9-68f3-4be3-97ec-4ade5b4aaa60
save_dir : ./FLAT_DIRECTORY
patch_save_dir : ./FLAT_DIRECTORY\patches
mask_save_dir : ./FLAT_DIRECTORY\masks
stitch_save_dir : ./FLAT_DIRECTORY\stitches
{'seg_params': {'seg_level': -1, 'sthresh': 8, 'mthresh': 7, 'close': 4, 'use_otsu': False, 'keep_ids': 'none', 'exclude_ids': 'none'}, 'filter_params': {'a_t': 100, 'a_h': 16, 'max_n_holes': 8}, 'patch_params': {'use_padding': True, 'contour_fn': 'four_pt'}, 'vis_params': {'vis_level': -1, 'line_thickness': 250}}
Computed patch_level = 0 for 40x WSI (target 20x)

100%|##########| 1/1 [00:04<00:00,  4.62s/it]


progress: 0.00, 0/1
processing TCGA-AY-4070-01Z-00-DX1.dd650

## Patches with custom method

In [37]:
import os
import h5py
import openslide
from PIL import Image
import numpy as np
import cv2
import csv

def get_white_pixel_ratio(pil_img, satThresh=30, valThresh=200):
    if pil_img.mode != 'RGB':
        pil_img = pil_img.convert('RGB')

    img_np = np.array(pil_img, dtype=np.uint8)
    hsv_img = cv2.cvtColor(img_np, cv2.COLOR_RGB2HSV)

    # Mask: low saturation AND high brightness (value)
    sat_mask = hsv_img[:, :, 1] < satThresh
    val_mask = hsv_img[:, :, 2] > valThresh
    white_mask = np.logical_and(sat_mask, val_mask)

    return np.sum(white_mask) / white_mask.size

def create_patches_from_h5(h5_path, svs_path, output_dir, skip_log_path):
    os.makedirs(output_dir, exist_ok=True)

    BASE_PATCH_SIZE = 2048
    SUB_PATCH_SIZE = 512
    base_name = os.path.splitext(os.path.basename(h5_path))[0]

    with h5py.File(h5_path, 'r') as h5_file:
        coords = h5_file['coords'][:]

    slide = openslide.OpenSlide(svs_path)
    patch_count = 0

    log_exists = os.path.exists(skip_log_path)
    
    # ✅ Open log file in write mode and keep it open during patch loop
    with open(skip_log_path, 'a', newline='') as log_file:
        writer = csv.writer(log_file)
        if not log_exists:
            writer.writerow(["PatchName", "X", "Y", "WhiteRatio"])  # Write header only once

        for i, (x, y) in enumerate(coords):
            base_patch = slide.read_region((int(x), int(y)), 0, (BASE_PATCH_SIZE, BASE_PATCH_SIZE)).convert('RGB')

            for row in range(4):
                for col in range(4):
                    offset_x = col * SUB_PATCH_SIZE
                    offset_y = row * SUB_PATCH_SIZE

                    abs_x = int(x) + offset_x
                    abs_y = int(y) + offset_y

                    sub_patch = base_patch.crop((
                        offset_x, offset_y,
                        offset_x + SUB_PATCH_SIZE, offset_y + SUB_PATCH_SIZE
                    ))

                    white_ratio = get_white_pixel_ratio(sub_patch)
                    patch_name = f"{base_name}_x{abs_x}_y{abs_y}_patch{patch_count:05d}.png"

                    # if white_ratio > 0.8:
                    #     writer.writerow([patch_name, abs_x, abs_y, round(white_ratio, 4)])
                    #     continue

                    sub_patch.save(os.path.join(output_dir, patch_name))
                    patch_count += 1

    print(f"✅ Done! Saved {patch_count} coordinate-named 512x512 patches to {output_dir}")

## Patches at 40x, each of 512 first and again 512

In [25]:
import os
import subprocess
from pathlib import Path

# Path to the .svs file to test
# svs_path = found_20x_path
svs_filename = os.path.basename(svs_path)
source_dir = os.path.dirname(svs_path)

# Label check (you must have this function and metadata file available)
bool_valid, label = check_svs_file(svs_filename, "./slides_metadata/slide_metadata_filtered.csv", filename_column="Filename", label_column="Label")
# if not bool_valid:
#     print("❌ SVS file is not valid or not labeled properly.")
# else:
#     print(f"✅ SVS file is valid. Label: {label}")
    
newnameofsvs = f"{svs_filename[:12]}_{label}"
flat_dir = "./FLAT_DIRECTORY"

# Create patches using CLAM script
command = [
    'python', './CLAM/create_patches_fp.py',
    '--source', source_dir,
    '--save_dir', flat_dir,
    '--patch_size', '512',
    '--step_size', '512',
    '--seg', '--patch'
]

print(f"🚀 Running patch extraction for: {svs_filename}")
with subprocess.Popen(command, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True) as proc:
    for line in proc.stdout:
        print(line, end='')

# Rename .h5 file
h5_path = rename_h5_file(svs_filename, label, h5_dir="./FLAT_DIRECTORY/patches")
print(f"✅ Renamed h5 file: {h5_path}")

# Generate skipped patch filename
skipped_csv = Path("./FLAT_DIRECTORY/patches") / f"{os.path.splitext(svs_filename)[0]}_skipped_patches.csv"

# Create PNG patches from h5
png_patch_output_dir = os.path.join("./Patches", newnameofsvs)
print(f"🎨 Generating PNG patches in: {png_patch_output_dir}")
create_patches_from_h5(h5_path, svs_path, png_patch_output_dir, skipped_csv)

🚀 Running patch extraction for: TCGA-AY-4070-01Z-00-DX1.dd650ac6-8480-4fd8-85b8-15a7840a5933.svs
source:  D:/Aamir Gulzar/WSI_Raw_data/021821c9-68f3-4be3-97ec-4ade5b4aaa60
patch_save_dir:  ./FLAT_DIRECTORY\patches
mask_save_dir:  ./FLAT_DIRECTORY\masks
stitch_save_dir:  ./FLAT_DIRECTORY\stitches
source : D:/Aamir Gulzar/WSI_Raw_data/021821c9-68f3-4be3-97ec-4ade5b4aaa60
save_dir : ./FLAT_DIRECTORY
patch_save_dir : ./FLAT_DIRECTORY\patches
mask_save_dir : ./FLAT_DIRECTORY\masks
stitch_save_dir : ./FLAT_DIRECTORY\stitches
{'seg_params': {'seg_level': -1, 'sthresh': 8, 'mthresh': 7, 'close': 4, 'use_otsu': False, 'keep_ids': 'none', 'exclude_ids': 'none'}, 'filter_params': {'a_t': 100, 'a_h': 16, 'max_n_holes': 8}, 'patch_params': {'use_padding': True, 'contour_fn': 'four_pt'}, 'vis_params': {'vis_level': -1, 'line_thickness': 250}}
Computed patch_level = 0 for 40x WSI (target 20x)

100%|##########| 1/1 [00:21<00:00, 21.83s/it]


progress: 0.00, 0/1
processing TCGA-AY-4070-01Z-00-DX1.dd650

## Patches at 20x, each of 1024 first and again 512

In [27]:
import os
import subprocess
from pathlib import Path

# Path to the .svs file to test
# svs_path = found_20x_path
svs_filename = os.path.basename(svs_path)
source_dir = os.path.dirname(svs_path)

# Label check (you must have this function and metadata file available)
bool_valid, label = check_svs_file(svs_filename, "./slides_metadata/slide_metadata_filtered.csv", filename_column="Filename", label_column="Label")
# if not bool_valid:
#     print("❌ SVS file is not valid or not labeled properly.")
# else:
#     print(f"✅ SVS file is valid. Label: {label}")
    
newnameofsvs = f"{svs_filename[:12]}_{label}"
flat_dir = "./FLAT_DIRECTORY"

# Create patches using CLAM script
command = [
    'python', './CLAM/create_patches_fp.py',
    '--source', source_dir,
    '--save_dir', flat_dir,
    '--patch_size', '1024',
    '--step_size', '1024',
    '--seg', '--patch'
]

print(f"🚀 Running patch extraction for: {svs_filename}")
with subprocess.Popen(command, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True) as proc:
    for line in proc.stdout:
        print(line, end='')

# Rename .h5 file
h5_path = rename_h5_file(svs_filename, label, h5_dir="./FLAT_DIRECTORY/patches")
print(f"✅ Renamed h5 file: {h5_path}")

# Generate skipped patch filename
skipped_csv = Path("./FLAT_DIRECTORY/patches") / f"{os.path.splitext(svs_filename)[0]}_skipped_patches.csv"

# Create PNG patches from h5
png_patch_output_dir = os.path.join("./Patches", newnameofsvs)
print(f"🎨 Generating PNG patches in: {png_patch_output_dir}")
create_patches_from_h5(h5_path, svs_path, png_patch_output_dir, skipped_csv)

🚀 Running patch extraction for: TCGA-AY-4070-01Z-00-DX1.dd650ac6-8480-4fd8-85b8-15a7840a5933.svs
source:  D:/Aamir Gulzar/WSI_Raw_data/021821c9-68f3-4be3-97ec-4ade5b4aaa60
patch_save_dir:  ./FLAT_DIRECTORY\patches
mask_save_dir:  ./FLAT_DIRECTORY\masks
stitch_save_dir:  ./FLAT_DIRECTORY\stitches
source : D:/Aamir Gulzar/WSI_Raw_data/021821c9-68f3-4be3-97ec-4ade5b4aaa60
save_dir : ./FLAT_DIRECTORY
patch_save_dir : ./FLAT_DIRECTORY\patches
mask_save_dir : ./FLAT_DIRECTORY\masks
stitch_save_dir : ./FLAT_DIRECTORY\stitches
{'seg_params': {'seg_level': -1, 'sthresh': 8, 'mthresh': 7, 'close': 4, 'use_otsu': False, 'keep_ids': 'none', 'exclude_ids': 'none'}, 'filter_params': {'a_t': 100, 'a_h': 16, 'max_n_holes': 8}, 'patch_params': {'use_padding': True, 'contour_fn': 'four_pt'}, 'vis_params': {'vis_level': -1, 'line_thickness': 250}}
Computed patch_level = 0 for 40x WSI (target 20x)

100%|##########| 1/1 [00:21<00:00, 21.38s/it]


progress: 0.00, 0/1
processing TCGA-AY-4070-01Z-00-DX1.dd650

## Patches at 20x, each of 1024 first and again 512 with white background filtering

In [35]:
create_patches_from_h5(h5_path, svs_path, png_patch_output_dir, skipped_csv)

✅ Done! Saved 1169 coordinate-named 512x512 patches to ./Patches\TCGA-AY-4070_nonMSIH


## Patches at 10x, each of 2048 first and again 512

In [38]:
import os
import subprocess
from pathlib import Path

# Path to the .svs file to test
# svs_path = found_20x_path
svs_filename = os.path.basename(svs_path)
source_dir = os.path.dirname(svs_path)

# Label check (you must have this function and metadata file available)
bool_valid, label = check_svs_file(svs_filename, "./slides_metadata/slide_metadata_filtered.csv", filename_column="Filename", label_column="Label")
# if not bool_valid:
#     print("❌ SVS file is not valid or not labeled properly.")
# else:
#     print(f"✅ SVS file is valid. Label: {label}")
    
newnameofsvs = f"{svs_filename[:12]}_{label}"
flat_dir = "./FLAT_DIRECTORY"

# Create patches using CLAM script
command = [
    'python', './CLAM/create_patches_fp.py',
    '--source', source_dir,
    '--save_dir', flat_dir,
    '--patch_size', '2048',
    '--step_size', '2048',
    '--seg', '--patch'
]

print(f"🚀 Running patch extraction for: {svs_filename}")
with subprocess.Popen(command, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True) as proc:
    for line in proc.stdout:
        print(line, end='')

# Rename .h5 file
h5_path = rename_h5_file(svs_filename, label, h5_dir="./FLAT_DIRECTORY/patches")
print(f"✅ Renamed h5 file: {h5_path}")

# Generate skipped patch filename
skipped_csv = Path("./FLAT_DIRECTORY/patches") / f"{os.path.splitext(svs_filename)[0]}_skipped_patches.csv"

# Create PNG patches from h5
png_patch_output_dir = os.path.join("./Patches", newnameofsvs)
print(f"🎨 Generating PNG patches in: {png_patch_output_dir}")
create_patches_from_h5(h5_path, svs_path, png_patch_output_dir, skipped_csv)

🚀 Running patch extraction for: TCGA-AY-4070-01Z-00-DX1.dd650ac6-8480-4fd8-85b8-15a7840a5933.svs
source:  D:/Aamir Gulzar/WSI_Raw_data/021821c9-68f3-4be3-97ec-4ade5b4aaa60
patch_save_dir:  ./FLAT_DIRECTORY\patches
mask_save_dir:  ./FLAT_DIRECTORY\masks
stitch_save_dir:  ./FLAT_DIRECTORY\stitches
source : D:/Aamir Gulzar/WSI_Raw_data/021821c9-68f3-4be3-97ec-4ade5b4aaa60
save_dir : ./FLAT_DIRECTORY
patch_save_dir : ./FLAT_DIRECTORY\patches
mask_save_dir : ./FLAT_DIRECTORY\masks
stitch_save_dir : ./FLAT_DIRECTORY\stitches
{'seg_params': {'seg_level': -1, 'sthresh': 8, 'mthresh': 7, 'close': 4, 'use_otsu': False, 'keep_ids': 'none', 'exclude_ids': 'none'}, 'filter_params': {'a_t': 100, 'a_h': 16, 'max_n_holes': 8}, 'patch_params': {'use_padding': True, 'contour_fn': 'four_pt'}, 'vis_params': {'vis_level': -1, 'line_thickness': 250}}
Computed patch_level = 0 for 40x WSI (target 20x)

100%|##########| 1/1 [00:22<00:00, 22.28s/it]


progress: 0.00, 0/1
processing TCGA-AY-4070-01Z-00-DX1.dd650

## creating patches of one 512x512 from 1024

In [42]:
def create_patches_from_h5(h5_path, svs_path, output_dir):
    os.makedirs(output_dir, exist_ok=True)

    PATCH_SIZE = 1024
    
    # Extract base name from h5 file (without extension)
    base_name = os.path.splitext(os.path.basename(h5_path))[0]
    
    # Load coordinates
    with h5py.File(h5_path, 'r') as h5_file:
        coords = h5_file['coords'][:]  # shape (N, 2), each row is (x, y)
    
    # Open the WSI
    slide = openslide.OpenSlide(svs_path)
    
    # Extract and save patches
    for i, (x, y) in enumerate(coords):
        patch = slide.read_region((int(x), int(y)), 0, (PATCH_SIZE, PATCH_SIZE)).convert('RGB')
        
        # Filename format: [basename]_x[y]_y[y]_patch[i].png
        patch_name = f"{base_name}_x{int(x)}_y{int(y)}_patch{i:05d}.png"
        patch.save(os.path.join(output_dir, patch_name))
    
    print(f"✅ Done! Saved {len(coords)} patches to {output_dir}")

In [43]:
import os
import subprocess
from pathlib import Path

# Path to the .svs file to test
# svs_path = found_20x_path
svs_filename = os.path.basename(svs_path)
source_dir = os.path.dirname(svs_path)

# Label check (you must have this function and metadata file available)
bool_valid, label = check_svs_file(svs_filename, "./slides_metadata/slide_metadata_filtered.csv", filename_column="Filename", label_column="Label")
# if not bool_valid:
#     print("❌ SVS file is not valid or not labeled properly.")
# else:
#     print(f"✅ SVS file is valid. Label: {label}")
    
newnameofsvs = f"{svs_filename[:12]}_{label}"
flat_dir = "./FLAT_DIRECTORY"

# Create patches using CLAM script
command = [
    'python', './CLAM/create_patches_fp.py',
    '--source', source_dir,
    '--save_dir', flat_dir,
    '--patch_size', '1024',
    '--step_size', '1024',
    '--seg', '--patch'
]

print(f"🚀 Running patch extraction for: {svs_filename}")
with subprocess.Popen(command, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True) as proc:
    for line in proc.stdout:
        print(line, end='')

# Rename .h5 file
h5_path = rename_h5_file(svs_filename, label, h5_dir="./FLAT_DIRECTORY/patches")
print(f"✅ Renamed h5 file: {h5_path}")

# Generate skipped patch filename
skipped_csv = Path("./FLAT_DIRECTORY/patches") / f"{os.path.splitext(svs_filename)[0]}_skipped_patches.csv"

# Create PNG patches from h5
png_patch_output_dir = os.path.join("./Patches", newnameofsvs)
print(f"🎨 Generating PNG patches in: {png_patch_output_dir}")
create_patches_from_h5(h5_path, svs_path, png_patch_output_dir)

🚀 Running patch extraction for: TCGA-AY-4070-01Z-00-DX1.dd650ac6-8480-4fd8-85b8-15a7840a5933.svs
source:  D:/Aamir Gulzar/WSI_Raw_data/021821c9-68f3-4be3-97ec-4ade5b4aaa60
patch_save_dir:  ./FLAT_DIRECTORY\patches
mask_save_dir:  ./FLAT_DIRECTORY\masks
stitch_save_dir:  ./FLAT_DIRECTORY\stitches
source : D:/Aamir Gulzar/WSI_Raw_data/021821c9-68f3-4be3-97ec-4ade5b4aaa60
save_dir : ./FLAT_DIRECTORY
patch_save_dir : ./FLAT_DIRECTORY\patches
mask_save_dir : ./FLAT_DIRECTORY\masks
stitch_save_dir : ./FLAT_DIRECTORY\stitches
{'seg_params': {'seg_level': -1, 'sthresh': 8, 'mthresh': 7, 'close': 4, 'use_otsu': False, 'keep_ids': 'none', 'exclude_ids': 'none'}, 'filter_params': {'a_t': 100, 'a_h': 16, 'max_n_holes': 8}, 'patch_params': {'use_padding': True, 'contour_fn': 'four_pt'}, 'vis_params': {'vis_level': -1, 'line_thickness': 250}}
Computed patch_level = 0 for 40x WSI (target 20x)

100%|##########| 1/1 [00:21<00:00, 21.21s/it]


progress: 0.00, 0/1
processing TCGA-AY-4070-01Z-00-DX1.dd650

In [44]:
from PIL import Image
import os
from tqdm import tqdm

# --- Paths ---
input_dir = './Patches/TCGA-AY-4070_nonMSIH/'   # e.g., "./patches_40x"
output_dir = './Patches/path_to_save_512x512_patches'    # e.g., "./patches_20x"

# Create output dir if not exists
os.makedirs(output_dir, exist_ok=True)

# --- Resize all patches ---
for fname in tqdm(os.listdir(input_dir)):
    if fname.endswith(".png"):
        input_path = os.path.join(input_dir, fname)
        output_path = os.path.join(output_dir, fname)

        # Open and resize image
        img = Image.open(input_path)
        img_resized = img.resize((512, 512), resample=Image.BICUBIC)

        # Save resized image
        img_resized.save(output_path)

100%|████████████████████████████████████████████████████████████████████████████████| 336/336 [00:21<00:00, 15.82it/s]


## Testing levels of downsample

In [ ]:
import os
import subprocess
from pathlib import Path

# Top-level directory containing subfolders with .svs files
wsi_root = "D:/Aamir Gulzar/WSI_Raw_data"
flat_dir = "./FLAT_DIRECTORY"
count=0

# Loop through all subfolders
for subdir in os.listdir(wsi_root):
    # if count==2:
    #     break
    # count+=1
    full_subdir_path = os.path.join(wsi_root, subdir)
    if not os.path.isdir(full_subdir_path):
        continue  # skip non-folder items

    # Search for .svs file in this subfolder
    svs_files = [f for f in os.listdir(full_subdir_path) if f.endswith('.svs')]
    if not svs_files:
        print(f"❌ No .svs file found in: {full_subdir_path}")
        continue
    else:
        print(svs_files)

    # If multiple .svs files exist, process each one
    for svs_file in svs_files:

        bool_valid, label= is_valid_svs_file(svs_file)

        # print(bool_valid)
        source_dir = full_subdir_path
        source_dir_quoted = f"{source_dir}"
        flat_dir_quoted = f"{flat_dir}"
        output_dir= "./FLAT_DIRECTORY/downsampled"
        target_size = (1024, 1024)
        svs_path = os.path.join(source_dir_quoted, svs_file)

        print(f"🚀 Processing: {svs_file} in {subdir}")
        level= downsample_and_resize_wsi(svs_path, output_dir, target_size, label)
        print(f"level: {level}\n")

In [ ]:
new_h5_path= rename_h5_file(svs_file, label, h5_dir="./FLAT_DIRECTORY/patches")
base_dir = "./FLAT_DIRECTORY"
sub_dir = "patches"

# Create full path
print(new_h5_path)
base_dir= "./Patches"
sub_dir= f"{svs_file[:12]}_{label}"
output_dir = os.path.join(base_dir, sub_dir) 
print(output_dir)
create_patches_from_h5(new_h5_path, svs_path, output_dir)

In [ ]:
# internal imports
from wsi_core.WholeSlideImage import WholeSlideImage
from wsi_core.wsi_utils import StitchCoords
from wsi_core.batch_process_utils import initialize_df
from Image_Crop import crop_stiched_image, crop_downsample_image
# other imports
import os
import numpy as np
import time
import argparse
import pdb
import pandas as pd
# The path can also be read from a config file, etc.
OPENSLIDE_PATH = r'D:\Aamir Gulzar\openslide-bin-4.0.0.3-windows-x64\bin'

if hasattr(os, 'add_dll_directory'):
    # Windows
    with os.add_dll_directory(OPENSLIDE_PATH):
        import openslide
else:
    import openslide
	
def stitching(file_path, wsi_object, downscale = 5):
	start = time.time()
	heatmap = StitchCoords(file_path, wsi_object, downscale=downscale, bg_color=(0,0,0), alpha=-1, draw_grid=False)
	total_time = time.time() - start
	return heatmap, total_time

def segment(WSI_object, seg_params, filter_params):
	### Start Seg Timer
	start_time = time.time()
	# Segment
	WSI_object.segmentTissue(**seg_params, filter_params=filter_params)
	### Stop Seg Timers
	seg_time_elapsed = time.time() - start_time   
	return WSI_object, seg_time_elapsed

def patching(WSI_object, **kwargs):
	### Start Patch Timer
	start_time = time.time()
	# Patch
	file_path = WSI_object.process_contours(**kwargs)
	### Stop Patch Timer
	patch_time_elapsed = time.time() - start_time
	return file_path, patch_time_elapsed

def get_wsi_magnification(slide):
    try:
        magnification = slide.properties[openslide.PROPERTY_NAME_OBJECTIVE_POWER]
        return float(magnification)
    except KeyError:
        raise ValueError("Magnification information not available in the WSI properties.")
	
def save_downsampled_cropped_image(WSI_object, slide_id, save_dir, seg_level):
    downsampled_image = WSI_object.getOpenSlide().read_region((0, 0), seg_level, WSI_object.level_dim[seg_level])
    downsampled_image = downsampled_image.convert("RGB")
    # downsampled_image_path = os.path.join(save_dir, slide_id + '_downsampled.png')
    # downsampled_image.save(downsampled_image_path)
    # Crop the downsampled image
    cropped_save_path = os.path.join(save_dir, slide_id + '.png')
    crop_downsample_image(downsampled_image,cropped_save_path)

def seg_and_patch(source, save_dir, patch_save_dir, mask_save_dir, stitch_save_dir, 
				  patch_size = 256, step_size = 256, 
				  seg_params = {'seg_level': -1, 'sthresh': 8, 'mthresh': 7, 'close': 4, 'use_otsu': False,
				  'keep_ids': 'none', 'exclude_ids': 'none'},
				  filter_params = {'a_t':100, 'a_h': 16, 'max_n_holes':8}, 
				  vis_params = {'vis_level': -1, 'line_thickness': 500},
				  patch_params = {'use_padding': True, 'contour_fn': 'four_pt'},
				  patch_level = 0,
				  use_default_params = False, 
				  seg = False, save_mask = True, 
				  stitch= False, 
				  patch = False, auto_skip=True, process_list = None):
	
	slides = sorted(os.listdir(source),reverse=True)
	slides = [slide for slide in slides if os.path.isfile(os.path.join(source, slide))]
	if process_list is None:
		df = initialize_df(slides, seg_params, filter_params, vis_params, patch_params)
	else:
		df = pd.read_csv(process_list)
		df = initialize_df(df, seg_params, filter_params, vis_params, patch_params)

	mask = df['process'] == 1
	process_stack = df[mask]

	total = len(process_stack)

	legacy_support = 'a' in df.keys()
	if legacy_support:
		print('detected legacy segmentation csv file, legacy support enabled')
		df = df.assign(**{'a_t': np.full((len(df)), int(filter_params['a_t']), dtype=np.uint32),
		'a_h': np.full((len(df)), int(filter_params['a_h']), dtype=np.uint32),
		'max_n_holes': np.full((len(df)), int(filter_params['max_n_holes']), dtype=np.uint32),
		'line_thickness': np.full((len(df)), int(vis_params['line_thickness']), dtype=np.uint32),
		'contour_fn': np.full((len(df)), patch_params['contour_fn'])})

	seg_times = 0.
	patch_times = 0.
	stitch_times = 0.

	for i in range(total):
		df.to_csv(os.path.join(save_dir, 'process_list_autogen.csv'), index=False)
		idx = process_stack.index[i]
		slide = process_stack.loc[idx, 'slide_id']
		print("\n\nprogress: {:.2f}, {}/{}".format(i/total, i, total))
		print('processing {}'.format(slide))
		
		df.loc[idx, 'process'] = 0
		slide_id, _ = os.path.splitext(slide)

		if auto_skip and os.path.isfile(os.path.join(patch_save_dir, slide_id + '.h5')):
			print('{} already exist in destination location, skipped'.format(slide_id))
			df.loc[idx, 'status'] = 'already_exist'
			continue

		# Inialize WSI
		full_path = os.path.join(source, slide)
		try:
			WSI_object = WholeSlideImage(full_path)
			mag = get_wsi_magnification(WSI_object.getOpenSlide())
			scale_factor = mag / 20
		except:
			continue
		
		# try:
		# 	WSI_object.initXML(os.path.splitext(full_path)[0]+'.xml')
		# except:
		# 	print('no xml annos found')
		# 	pass
		# try:
		# 	WSI_object.initTxt(os.path.splitext(full_path)[0]+'.txt')
		# except:
		# 	pass

		if use_default_params:
			current_vis_params = vis_params.copy()
			current_filter_params = filter_params.copy()
			current_seg_params = seg_params.copy()
			current_patch_params = patch_params.copy()
		else:
			current_vis_params = {}
			current_filter_params = {}
			current_seg_params = {}
			current_patch_params = {}

			for key in vis_params.keys():
				if legacy_support and key == 'vis_level':
					df.loc[idx, key] = -1
				current_vis_params.update({key: df.loc[idx, key]})

			for key in filter_params.keys():
				if legacy_support and key == 'a_t':
					old_area = df.loc[idx, 'a']
					seg_level = df.loc[idx, 'seg_level']
					scale = WSI_object.level_downsamples[seg_level]
					adjusted_area = int(old_area * (scale[0] * scale[1]) / (512 * 512))
					current_filter_params.update({key: adjusted_area})
					df.loc[idx, key] = adjusted_area
				current_filter_params.update({key: df.loc[idx, key]})

			for key in seg_params.keys():
				if legacy_support and key == 'seg_level':
					df.loc[idx, key] = -1
				current_seg_params.update({key: df.loc[idx, key]})

			for key in patch_params.keys():
				current_patch_params.update({key: df.loc[idx, key]})

		if current_vis_params['vis_level'] < 0:
			# pdb.set_trace()

			if len(WSI_object.level_dim) == 1:
				current_vis_params['vis_level'] = 0
			else:
				wsi = WSI_object.getOpenSlide()
				best_level = wsi.get_best_level_for_downsample(scale_factor)
				current_vis_params['vis_level'] = best_level

		if current_seg_params['seg_level'] < 0:
			if len(WSI_object.level_dim) == 1:
				current_seg_params['seg_level'] = 0
	
			else:
				wsi = WSI_object.getOpenSlide()
				best_level = wsi.get_best_level_for_downsample(scale_factor)
				current_seg_params['seg_level'] = best_level

		keep_ids = str(current_seg_params['keep_ids'])
		if keep_ids != 'none' and len(keep_ids) > 0:
			str_ids = current_seg_params['keep_ids']
			current_seg_params['keep_ids'] = np.array(str_ids.split(',')).astype(int)
		else:
			current_seg_params['keep_ids'] = []

		exclude_ids = str(current_seg_params['exclude_ids'])
		if exclude_ids != 'none' and len(exclude_ids) > 0:
			str_ids = current_seg_params['exclude_ids']
			current_seg_params['exclude_ids'] = np.array(str_ids.split(',')).astype(int)
		else:
			current_seg_params['exclude_ids'] = []

		w, h = WSI_object.level_dim[current_seg_params['seg_level']] 
		if w * h > 7e9:
			print('level_dim {} x {} is likely too large for successful segmentation, aborting'.format(w, h))
			df.loc[idx, 'status'] = 'failed_seg'
			continue

		df.loc[idx, 'vis_level'] = current_vis_params['vis_level']
		df.loc[idx, 'seg_level'] = current_seg_params['seg_level']

		seg_time_elapsed = -1
		if seg:
			WSI_object, seg_time_elapsed = segment(WSI_object, current_seg_params, current_filter_params) 

		if save_mask:
			mask = WSI_object.visWSI(**current_vis_params)
			mask_path = os.path.join(mask_save_dir, slide_id+'.png')
			mask.save(mask_path)

		patch_time_elapsed = -1 # Default time
		# import pdb;pdb.set_trace()
		if patch:
			current_patch_params.update({'patch_level': patch_level, 'patch_size': patch_size, 'step_size': step_size, 
										 'save_path': patch_save_dir})
			# file_path, patch_time_elapsed = patching(WSI_object = WSI_object,  **current_patch_params,)

		stitch_time_elapsed = -1
		if stitch:
			file_path = os.path.join(patch_save_dir, slide_id+'.h5')
			if os.path.isfile(file_path):
				heatmap, stitch_time_elapsed = stitching(file_path, WSI_object, downscale=scale_factor)
				# stitch_path = os.path.join(stitch_save_dir, slide_id+'.png')
				# heatmap.save(stitch_path)
				# print("heatmap after stitching : ",heatmap)
				stitched_cropped_path=r'E:\\Aamir Gulzar\\dataset\\stitched_cropped_data'
				if not os.path.exists(stitched_cropped_path):
					os.makedirs(stitched_cropped_path)
				stitched_cropped_path = os.path.join(stitched_cropped_path, slide_id+'.png')
				crop_stiched_image(heatmap, stitched_cropped_path)

		print("segmentation took {} seconds".format(seg_time_elapsed))
		print("patching took {} seconds".format(patch_time_elapsed))
		print("stitching took {} seconds".format(stitch_time_elapsed))
		df.loc[idx, 'status'] = 'processed'

		seg_times += seg_time_elapsed
		patch_times += patch_time_elapsed
		stitch_times += stitch_time_elapsed

	seg_times /= total
	patch_times /= total
	stitch_times /= total

	df.to_csv(os.path.join(save_dir, 'process_list_autogen.csv'), index=False)
	print("average segmentation time in s per slide: {}".format(seg_times))
	print("average patching time in s per slide: {}".format(patch_times))
	print("average stiching time in s per slide: {}".format(stitch_times))
		
	return seg_times, patch_times

parser = argparse.ArgumentParser(description='seg and patch')
parser.add_argument('--source', type = str,
					help='path to folder containing raw wsi image files')
parser.add_argument('--step_size', type = int, default=256,
					help='step_size')
parser.add_argument('--patch_size', type = int, default=256,
					help='patch_size')
parser.add_argument('--patch', default=False, action='store_true')
parser.add_argument('--seg', default=False, action='store_true')
parser.add_argument('--stitch', default=False, action='store_true')
parser.add_argument('--no_auto_skip', default=True, action='store_false')
parser.add_argument('--save_dir', type = str,
					help='directory to save processed data')
parser.add_argument('--preset', default=None, type=str,
					help='predefined profile of default segmentation and filter parameters (.csv)')
parser.add_argument('--patch_level', type=int, default=0, 
					help='downsample level at which to patch')
parser.add_argument('--process_list',  type = str, default=None,
					help='name of list of images to process with parameters (.csv)')
parser.add_argument('--use_ostu', default=False, action='store_true')

if __name__ == '__main__':
	args = parser.parse_args()

	patch_save_dir = os.path.join(args.save_dir, 'patches')
	mask_save_dir = os.path.join(args.save_dir, 'masks')
	stitch_save_dir = os.path.join(args.save_dir, 'stitches')

	if args.process_list:
		process_list = os.path.join(args.save_dir, args.process_list)

	else:
		process_list = None

	print('source: ', args.source)
	print('patch_save_dir: ', patch_save_dir)
	print('mask_save_dir: ', mask_save_dir)
	print('stitch_save_dir: ', stitch_save_dir)
	
	directories = {'source': args.source, 
				   'save_dir': args.save_dir,
				   'patch_save_dir': patch_save_dir, 
				   'mask_save_dir' : mask_save_dir, 
				   'stitch_save_dir': stitch_save_dir} 

	for key, val in directories.items():
		print("{} : {}".format(key, val))
		if key not in ['source']:
			os.makedirs(val, exist_ok=True)

	seg_params = {'seg_level': -1, 'sthresh': 8, 'mthresh': 7, 'close': 4, 'use_otsu': args.use_ostu,
				  'keep_ids': 'none', 'exclude_ids': 'none'}
	filter_params = {'a_t':100, 'a_h': 16, 'max_n_holes':8}
	vis_params = {'vis_level': -1, 'line_thickness': 250}
	patch_params = {'use_padding': True, 'contour_fn': 'four_pt'}

	if args.preset:
		preset_df = pd.read_csv(os.path.join('presets', args.preset))
		for key in seg_params.keys():
			seg_params[key] = preset_df.loc[0, key]

		for key in filter_params.keys():
			filter_params[key] = preset_df.loc[0, key]

		for key in vis_params.keys():
			vis_params[key] = preset_df.loc[0, key]

		for key in patch_params.keys():
			patch_params[key] = preset_df.loc[0, key]
	
	parameters = {'seg_params': seg_params,
				  'filter_params': filter_params,
	 			  'patch_params': patch_params,
				  'vis_params': vis_params}

	print(parameters)

	seg_times, patch_times = seg_and_patch(**directories, **parameters,
											patch_size = args.patch_size, step_size=args.step_size, 
											seg = args.seg,  use_default_params=False, save_mask = False, 
											stitch= args.stitch,
											patch_level=args.patch_level, patch = args.patch,
											process_list = process_list, auto_skip=args.no_auto_skip)


In [ ]:
# In the seg_and_patch function...

for i in tqdm(range(total)):
    # ... (existing code to get slide_id, etc.) ...

    # Inialize WSI
    full_path = os.path.join(source, slide)
    WSI_object = WholeSlideImage(full_path)
   
    # =======================================================================
    # START: New logic to dynamically set patch_level
    # =======================================================================
   
    # Default to the command-line argument
    effective_patch_level = patch_level
   
    try:
        # Get the underlying openslide object to access its properties
        wsi = WSI_object.getOpenSlide()
       
        # The 'objective-power' property stores the magnification (e.g., '20' or '40')
        objective_power = int(wsi.properties.get('openslide.objective-power'))
        print(f"\nINFO: Slide '{slide_id}' has a native magnification of {objective_power}x.")
       
        # If magnification is greater than 20x (e.g., 40x), we want to downsample it.
        # We'll use level 1, which is typically a 2x downsample.
        if objective_power > 20:
            effective_patch_level = 1
            print(f"-> High magnification detected. Dynamically setting patch_level to {effective_patch_level} to get ~20x.")
       
        # If magnification is 20x or less, we patch at the native resolution.
        else:
            effective_patch_level = 0
            print(f"-> Magnification is <= 20x. Dynamically setting patch_level to {effective_patch_level} to use native resolution.")

    except Exception as e:
        print(f"\nWARNING: Could not determine objective power for '{slide_id}'. Reason: {e}")
        print(f"-> Falling back to the user-provided patch_level: {patch_level}")
        effective_patch_level = patch_level

    # =======================================================================
    # END: New logic
    # =======================================================================

    # ... (existing code to set up params) ...

    # ... (inside the 'if patch:' block, make sure to use the new variable)
    if patch:
        current_patch_params.update({'patch_level': effective_patch_level, # <-- Use the dynamic level
                                     'patch_size': patch_size, 'step_size': step_size,
                                     'save_path': patch_save_dir})
        file_path, patch_time_elapsed = patching(WSI_object = WSI_object,  **current_patch_params,)
   
    # ... (rest of the loop) ...


In [ ]:
import os
import shutil
from pathlib import Path

source_dir = "D:/Aamir Gulzar/WSI_Raw_data/00d69f69-3120-4b0c-8985-aab4edf9bc4b"
flat_dir = "./FLAT_DIRECTORY"
os.makedirs(flat_dir, exist_ok=True)

for root, _, files in os.walk(source_dir):
    for file in files:
        if file.endswith(".svs"):
            src_path = os.path.join(root, file)
            dest_path = os.path.join(flat_dir, f"{Path(root).name}_{file}")
            shutil.copy(src_path, dest_path) # or use os.symlink for symlinks

print("Flattened directory ready:", flat_dir)

In [ ]:
# !python "./CLAM/create_patches_fp.py" --source f'{source_dir}' --save_dir f'{flat_dir}' --patch_size 512 --seg --patch --stitch

In [ ]:
# Wrap paths containing spaces in quotes only when running the subprocess
import subprocess

source_dir_quoted = f'{source_dir}'
flat_dir_quoted = f'{flat_dir}'

# Run the create_patches_fp.py script with subprocess
command = [
    'python', './CLAM/create_patches_fp.py', 
    '--source', source_dir_quoted, 
    '--save_dir', flat_dir_quoted, 
    '--patch_size', '256', 
    '--seg', '--patch', '--stitch'
]

# Execute the command
result = subprocess.run(command, capture_output=True, text=True)

# Check for errors or success
if result.returncode == 0:
    print("Successfully processed patches.")
else:
    print(f"Error: {result.stderr}")


In [67]:
import h5py

# Open the HDF5 file in read mode
with h5py.File('./FLAT_DIRECTORY/patches/TCGA-D5-5539-01Z-00-DX1.9c46fe78-2adb-4f49-9141-cda135c2c90b.h5', 'r') as f:
    # List all the top-level keys (groups or datasets)
    print(f.keys())  # This will print the list of group/dataset names in the root
    
    # Explore groups and datasets
    print(len(f.keys()))
    for key in f.keys():
        print(f"Group or Dataset name: {key}")
        # Get the type of the key (it could be a group or a dataset)
        print(f"Type of {key}: {type(f[key])}")

        dataset = f[key]
        
        # Read the dataset as a numpy array
        data = dataset[:]
        print(data)
        
        # Or you can access a specific slice of the dataset
        data_slice = dataset[10:20, 10:20]  # for a 2D dataset
        print(data_slice)

FileNotFoundError: [Errno 2] Unable to synchronously open file (unable to open file: name = './FLAT_DIRECTORY/patches/TCGA-D5-5539-01Z-00-DX1.9c46fe78-2adb-4f49-9141-cda135c2c90b.h5', errno = 2, error message = 'No such file or directory', flags = 0, o_flags = 0)

In [ ]:
with h5py.File('path_to_file.h5', 'r') as f:
    # Access a dataset by its name
    dataset = f['dataset_name']
    
    # Read the dataset as a numpy array
    data = dataset[:]
    print(data)
    
    # Or you can access a specific slice of the dataset
    data_slice = dataset[10:20, 10:20]  # for a 2D dataset
    print(data_slice)

In [ ]:
import h5py

h5_file_path = './FLAT_DIRECTORY/patches/TCGA-D5-5539-01Z-00-DX1.9c46fe78-2adb-4f49-9141-cda135c2c90b.h5'

def explore_h5_structure(path):
    with h5py.File(path, 'r') as f:
        def print_attrs(name, obj):
            print(f"{name}: {type(obj)}, shape = {getattr(obj, 'shape', 'N/A')}")
        f.visititems(print_attrs)

explore_h5_structure(h5_file_path)


In [ ]:
import h5py
import openslide
from PIL import Image
import numpy as np
import os

# Paths
h5_path = './FLAT_DIRECTORY/patches/TCGA-D5-5539-01Z-00-DX1.9c46fe78-2adb-4f49-9141-cda135c2c90b.h5'
svs_path = './FLAT_DIRECTORY/00d69f69-3120-4b0c-8985-aab4edf9bc4b_TCGA-D5-5539-01Z-00-DX1.9c46fe78-2adb-4f49-9141-cda135c2c90b.svs'  # or .tif
output_dir = './patches_png/'
os.makedirs(output_dir, exist_ok=True)

# Patch size (adjust if needed)
PATCH_SIZE = 256

# Load coordinates
with h5py.File(h5_path, 'r') as h5_file:
    coords = h5_file['coords'][:]  # shape (N, 2), each row is (x, y)

# Open the whole-slide image
slide = openslide.OpenSlide(svs_path)

# Extract and save patches
for i, (x, y) in enumerate(coords):
    patch = slide.read_region((int(x), int(y)), 0, (PATCH_SIZE, PATCH_SIZE)).convert('RGB')
    patch.save(os.path.join(output_dir, f'patch_{i:05d}.png'))

print(f"✅ Done! Saved {len(coords)} patches to {output_dir}")

In [ ]:
import openslide
from PIL import Image
import os
import glob

# # Configuration
# wsi_root = "D:/Aamir Gulzar/WSI_Raw_data"
# flat_dir = "./FLAT_DIRECTORY"

# main_directory = "./FLAT_DIRECTORY"  # Change this to your main directory
# svs_filename = "TCGA-AG-3732"  # Change this to the filename you're looking for (without extension)

# # Function to find .svs file in directory (including subfolders)
# def find_svs_file(directory, filename):
#     """
#     Find a .svs file in the given directory and all subfolders that starts with the given filename
#     """
#     search_pattern = os.path.join(directory, "**", f"{filename}*.svs")
#     matching_files = glob.glob(search_pattern, recursive=True)
    
#     if not matching_files:
#         raise FileNotFoundError(f"No .svs file found starting with '{filename}' in directory '{directory}' or its subfolders")
#     elif len(matching_files) > 1:
#         print(f"Multiple .svs files found: {matching_files}")
#         print(f"Using the first one: {matching_files[0]}")
    
#     return matching_files[0]

# Find the .svs file automatically
# try:
#     svs_path = find_svs_file(main_directory, svs_filename)
#     print(f"Found .svs file: {svs_path}")
# except FileNotFoundError as e:
#     print(e)
#     exit(1)

# Load the slide
slide = openslide.OpenSlide(svs_path)

# Print basic info
print("Levels:", slide.level_count)
print("Level dimensions:", slide.level_dimensions)
print("Level downsamples:", slide.level_downsamples)

# MANUAL METHOD - Downsample to 20x
slide = openslide.OpenSlide(svs_path)

# Get the full-resolution image (assumed to be 40x)
full_image = slide.read_region((0, 0), 0, slide.dimensions).convert("RGB")

# Downsample to 20x by resizing to 50%
new_size = (slide.dimensions[0] // 2, slide.dimensions[1] // 2)
downsampled_20x = full_image.resize(new_size, Image.LANCZOS)

# Create output filename using first 12 characters of the .svs filename
svs_base_filename = os.path.splitext(os.path.basename(svs_path))[0]
output_filename = f"{svs_base_filename[:12]}_downsampled_20x.png"
downsampled_20x.save(output_filename)
print(f"20x image saved as {output_filename}")

slide.close()

In [ ]:
svs_path

In [ ]:
# internal imports
from wsi_core.WholeSlideImage import WholeSlideImage
from wsi_core.wsi_utils import StitchCoords
from wsi_core.batch_process_utils import initialize_df
# other imports
import os
import numpy as np
import time
import argparse
import pdb
import pandas as pd
from tqdm import tqdm

OPENSLIDE_PATH = r'D:\Aamir Gulzar\openslide-bin-4.0.0.3-windows-x64\bin'
if hasattr(os, 'add_dll_directory'):
    # Windows
    with os.add_dll_directory(OPENSLIDE_PATH):
        import openslide
else:
    import openslide

def stitching(file_path, wsi_object, downscale = 64):
	start = time.time()
	heatmap = StitchCoords(file_path, wsi_object, downscale=downscale, bg_color=(0,0,0), alpha=-1, draw_grid=False)
	total_time = time.time() - start
	
	return heatmap, total_time

def segment(WSI_object, seg_params = None, filter_params = None, mask_file = None):
	### Start Seg Timer
	start_time = time.time()
	# Use segmentation file
	if mask_file is not None:
		WSI_object.initSegmentation(mask_file)
	# Segment	
	else:
		WSI_object.segmentTissue(**seg_params, filter_params=filter_params)

	### Stop Seg Timers
	seg_time_elapsed = time.time() - start_time   
	return WSI_object, seg_time_elapsed

def patching(WSI_object, **kwargs):
	### Start Patch Timer
	start_time = time.time()

	# Patch
	file_path = WSI_object.process_contours(**kwargs)


	### Stop Patch Timer
	patch_time_elapsed = time.time() - start_time
	return file_path, patch_time_elapsed


def seg_and_patch(source, save_dir, patch_save_dir, mask_save_dir, stitch_save_dir, 
				  patch_size = 256, step_size = 256, 
				  seg_params = {'seg_level': -1, 'sthresh': 8, 'mthresh': 7, 'close': 4, 'use_otsu': False,
				  'keep_ids': 'none', 'exclude_ids': 'none'},
				  filter_params = {'a_t':100, 'a_h': 16, 'max_n_holes':8}, 
				  vis_params = {'vis_level': -1, 'line_thickness': 500},
				  patch_params = {'use_padding': True, 'contour_fn': 'four_pt'},
				  patch_level = 0,
				  use_default_params = False, 
				  seg = False, save_mask = False, 
				  stitch= False, 
				  patch = False, auto_skip=True, process_list = None):
	


	slides = sorted(os.listdir(source))
	slides = [slide for slide in slides if os.path.isfile(os.path.join(source, slide))]
	if process_list is None:
		df = initialize_df(slides, seg_params, filter_params, vis_params, patch_params)
	
	else:
		df = pd.read_csv(process_list)
		df = initialize_df(df, seg_params, filter_params, vis_params, patch_params)

	mask = df['process'] == 1
	process_stack = df[mask]

	total = len(process_stack)

	legacy_support = 'a' in df.keys()
	if legacy_support:
		print('detected legacy segmentation csv file, legacy support enabled')
		df = df.assign(**{'a_t': np.full((len(df)), int(filter_params['a_t']), dtype=np.uint32),
		'a_h': np.full((len(df)), int(filter_params['a_h']), dtype=np.uint32),
		'max_n_holes': np.full((len(df)), int(filter_params['max_n_holes']), dtype=np.uint32),
		'line_thickness': np.full((len(df)), int(vis_params['line_thickness']), dtype=np.uint32),
		'contour_fn': np.full((len(df)), patch_params['contour_fn'])})

	seg_times = 0.
	patch_times = 0.
	stitch_times = 0.

	for i in tqdm(range(total)):
		df.to_csv(os.path.join(save_dir, 'process_list_autogen.csv'), index=False)
		idx = process_stack.index[i]
		slide = process_stack.loc[idx, 'slide_id']
		print("\n\nprogress: {:.2f}, {}/{}".format(i/total, i, total))
		print('processing {}'.format(slide))
		
		df.loc[idx, 'process'] = 0
		slide_id, _ = os.path.splitext(slide)
		# slide_id = slide_id[:12]

		if auto_skip and os.path.isfile(os.path.join(patch_save_dir, slide_id + '.h5')):
			print('{} already exist in destination location, skipped'.format(slide_id))
			df.loc[idx, 'status'] = 'already_exist'
			continue

		# Inialize WSI
		full_path = os.path.join(source, slide)
		WSI_object = WholeSlideImage(full_path)

		if use_default_params:
			current_vis_params = vis_params.copy()
			current_filter_params = filter_params.copy()
			current_seg_params = seg_params.copy()
			current_patch_params = patch_params.copy()
			
		else:
			current_vis_params = {}
			current_filter_params = {}
			current_seg_params = {}
			current_patch_params = {}


			for key in vis_params.keys():
				if legacy_support and key == 'vis_level':
					df.loc[idx, key] = -1
				current_vis_params.update({key: df.loc[idx, key]})

			for key in filter_params.keys():
				if legacy_support and key == 'a_t':
					old_area = df.loc[idx, 'a']
					seg_level = df.loc[idx, 'seg_level']
					scale = WSI_object.level_downsamples[seg_level]
					adjusted_area = int(old_area * (scale[0] * scale[1]) / (512 * 512))
					current_filter_params.update({key: adjusted_area})
					df.loc[idx, key] = adjusted_area
				current_filter_params.update({key: df.loc[idx, key]})

			for key in seg_params.keys():
				if legacy_support and key == 'seg_level':
					df.loc[idx, key] = -1
				current_seg_params.update({key: df.loc[idx, key]})

			for key in patch_params.keys():
				current_patch_params.update({key: df.loc[idx, key]})

		if current_vis_params['vis_level'] < 0:
			if len(WSI_object.level_dim) == 1:
				current_vis_params['vis_level'] = 0
			
			else:	
				wsi = WSI_object.getOpenSlide()
				best_level = wsi.get_best_level_for_downsample(64)
				current_vis_params['vis_level'] = best_level

		if current_seg_params['seg_level'] < 0:
			if len(WSI_object.level_dim) == 1:
				current_seg_params['seg_level'] = 0
			
			else:
				wsi = WSI_object.getOpenSlide()
				best_level = wsi.get_best_level_for_downsample(64)
				current_seg_params['seg_level'] = best_level

		keep_ids = str(current_seg_params['keep_ids'])
		if keep_ids != 'none' and len(keep_ids) > 0:
			str_ids = current_seg_params['keep_ids']
			current_seg_params['keep_ids'] = np.array(str_ids.split(',')).astype(int)
		else:
			current_seg_params['keep_ids'] = []

		exclude_ids = str(current_seg_params['exclude_ids'])
		if exclude_ids != 'none' and len(exclude_ids) > 0:
			str_ids = current_seg_params['exclude_ids']
			current_seg_params['exclude_ids'] = np.array(str_ids.split(',')).astype(int)
		else:
			current_seg_params['exclude_ids'] = []

		w, h = WSI_object.level_dim[current_seg_params['seg_level']] 
		if w * h > 1e8:
			print('level_dim {} x {} is likely too large for successful segmentation, aborting'.format(w, h))
			df.loc[idx, 'status'] = 'failed_seg'
			continue

		df.loc[idx, 'vis_level'] = current_vis_params['vis_level']
		df.loc[idx, 'seg_level'] = current_seg_params['seg_level']


		seg_time_elapsed = -1
		if seg:
			WSI_object, seg_time_elapsed = segment(WSI_object, current_seg_params, current_filter_params) 

		if save_mask:
			mask = WSI_object.visWSI(**current_vis_params)
			mask_path = os.path.join(mask_save_dir, slide_id+'.jpg')
			mask.save(mask_path)

		patch_time_elapsed = -1 # Default time
		if patch:
			current_patch_params.update({'patch_level': patch_level, 'patch_size': patch_size, 'step_size': step_size, 
										 'save_path': patch_save_dir})
			file_path, patch_time_elapsed = patching(WSI_object = WSI_object,  **current_patch_params,)
		
		stitch_time_elapsed = -1
		if stitch:
			file_path = os.path.join(patch_save_dir, slide_id+'.h5')
			if os.path.isfile(file_path):
				heatmap, stitch_time_elapsed = stitching(file_path, WSI_object, downscale=64)
				stitch_path = os.path.join(stitch_save_dir, slide_id+'.jpg')
				heatmap.save(stitch_path)

		print("segmentation took {} seconds".format(seg_time_elapsed))
		print("patching took {} seconds".format(patch_time_elapsed))
		print("stitching took {} seconds".format(stitch_time_elapsed))
		df.loc[idx, 'status'] = 'processed'

		seg_times += seg_time_elapsed
		patch_times += patch_time_elapsed
		stitch_times += stitch_time_elapsed

	seg_times /= total
	patch_times /= total
	stitch_times /= total

	df.to_csv(os.path.join(save_dir, 'process_list_autogen.csv'), index=False)
	print("average segmentation time in s per slide: {}".format(seg_times))
	print("average patching time in s per slide: {}".format(patch_times))
	print("average stiching time in s per slide: {}".format(stitch_times))
		
	return seg_times, patch_times

parser = argparse.ArgumentParser(description='seg and patch')
parser.add_argument('--source', type = str,
					help='path to folder containing raw wsi image files')
parser.add_argument('--step_size', type = int, default=256,
					help='step_size')
parser.add_argument('--patch_size', type = int, default=256,
					help='patch_size')
parser.add_argument('--patch', default=False, action='store_true')
parser.add_argument('--seg', default=False, action='store_true')
parser.add_argument('--stitch', default=False, action='store_true')
parser.add_argument('--no_auto_skip', default=True, action='store_false')
parser.add_argument('--save_dir', type = str,
					help='directory to save processed data')
parser.add_argument('--preset', default=None, type=str,
					help='predefined profile of default segmentation and filter parameters (.csv)')
parser.add_argument('--patch_level', type=int, default=0, 
					help='downsample level at which to patch')
parser.add_argument('--process_list',  type = str, default=None,
					help='name of list of images to process with parameters (.csv)')

if __name__ == '__main__':
	args = parser.parse_args()

	patch_save_dir = os.path.join(args.save_dir, 'patches')
	mask_save_dir = os.path.join(args.save_dir, 'masks')
	stitch_save_dir = os.path.join(args.save_dir, 'stitches')

	if args.process_list:
		process_list = os.path.join(args.save_dir, args.process_list)

	else:
		process_list = None

	print('source: ', args.source)
	print('patch_save_dir: ', patch_save_dir)
	print('mask_save_dir: ', mask_save_dir)
	print('stitch_save_dir: ', stitch_save_dir)
	
	directories = {'source': args.source, 
				   'save_dir': args.save_dir,
				   'patch_save_dir': patch_save_dir, 
				   'mask_save_dir' : mask_save_dir, 
				   'stitch_save_dir': stitch_save_dir} 

	for key, val in directories.items():
		print("{} : {}".format(key, val))
		if key not in ['source']:
			os.makedirs(val, exist_ok=True)

	seg_params = {'seg_level': -1, 'sthresh': 8, 'mthresh': 7, 'close': 4, 'use_otsu': False,
				  'keep_ids': 'none', 'exclude_ids': 'none'}
	filter_params = {'a_t':100, 'a_h': 16, 'max_n_holes':8}
	vis_params = {'vis_level': -1, 'line_thickness': 250}
	patch_params = {'use_padding': True, 'contour_fn': 'four_pt'}

	if args.preset:
		preset_df = pd.read_csv(os.path.join('presets', args.preset))
		for key in seg_params.keys():
			seg_params[key] = preset_df.loc[0, key]

		for key in filter_params.keys():
			filter_params[key] = preset_df.loc[0, key]

		for key in vis_params.keys():
			vis_params[key] = preset_df.loc[0, key]

		for key in patch_params.keys():
			patch_params[key] = preset_df.loc[0, key]
	
	parameters = {'seg_params': seg_params,
				  'filter_params': filter_params,
	 			  'patch_params': patch_params,
				  'vis_params': vis_params}

	print(parameters)

	try:
		# Pick a representative SVS file from the source folder
		sample_files = [f for f in os.listdir(args.source) if f.endswith('.svs')]
		if not sample_files:
			raise FileNotFoundError("No .svs files found in source directory.")

		sample_slide_path = os.path.join(args.source, sample_files[0])
		slide = openslide.OpenSlide(sample_slide_path)

		# Get WSI magnification and compute scale factor
		mag = int(slide.properties.get("openslide.objective-power", 20))  # fallback to 20x
		scale_factor = mag / 20  # e.g., 40 / 20 = 2
		patch_level = slide.get_best_level_for_downsample(scale_factor)

		print(f"Computed patch_level = {patch_level} for {mag}x WSI (target 20x)")
		args.patch_level = patch_level  # override the patch_level from CLI
	except Exception as e:
		print(f"Failed to determine patch_level automatically: {e}")
		print(f"Using default patch_level = {args.patch_level}")

	seg_times, patch_times = seg_and_patch(**directories, **parameters,
											patch_size = args.patch_size, step_size=args.step_size, 
											seg = args.seg,  use_default_params=False, save_mask = False, 
											stitch= args.stitch,
											patch_level=args.patch_level, patch = args.patch,
											process_list = process_list, auto_skip=args.no_auto_skip)


In [ ]:
import openslide
import os
from PIL import Image

def get_wsi_magnification(slide, default_power=20):
    """
    Get the magnification of a WSI image using OpenSlide properties.
    """
    objective_power = slide.properties.get("openslide.objective-power")
    if objective_power is not None:
        return int(objective_power)
    else:
        print("Warning: Objective power not found. Using default:", default_power)
        return default_power

def downsample_and_resize_wsi(wsi_path, output_dir, target_size, label):
    slide = openslide.OpenSlide(wsi_path)
    mag = get_wsi_magnification(slide)
    
    print(f"Magnification: {mag}x")
    print("Levels:", slide.level_count)
    print("Level dimensions:", slide.level_dimensions)
    print("Level downsamples:", slide.level_downsamples)

    print("slide.level_dimensions", slide.level_dimensions[1])
    newdim = (slide.level_dimensions[1][0] // 2, slide.level_dimensions[0][1] // 2)
    print(f"New dimension: {newdim}")
    # Change here: target downsampling to 20x instead of 5x
    scale_factor = mag / 20

    print(f"Scale factor (to downsample to 20x): {scale_factor}")
    
    level = slide.get_best_level_for_downsample(scale_factor)
    downsample = slide.level_downsamples[level]
    
    print(f"Using level {level} with downsample factor {downsample}")
    
    # # Read the region at the chosen level dimensions
    # region = slide.read_region((0, 0), level, slide.level_dimensions[level]).convert("RGB")
    
    # # Calculate how much to resize manually to get exactly 20x
    # manual_scale = downsample / scale_factor  # e.g. if level downsample=4, scale_factor=2, manual_scale=2
    
    # new_width = int(region.width / manual_scale)
    # new_height = int(region.height / manual_scale)
    
    # print(f"Resizing region from {region.size} to ({new_width}, {new_height})")
    
    # downsampled_img = region.resize((new_width, new_height), Image.LANCZOS)
    
    # # Save image
    # output_path = os.path.join(output_dir, f"{os.path.basename(wsi_path)[:12]}_{label}.png")
    # os.makedirs(os.path.dirname(output_path), exist_ok=True)
    # downsampled_img.save(output_path)
    # print(f"Saved downsampled WSI to {output_path}")
    return level

In [ ]:
import openslide
from PIL import Image
import numpy as np
import tifffile as tiff
import os

# def downsample_wsi_to_20x_tiff(svs_path, tiff_output_path, tile_size=1024, downsample_factor=2):
#     slide = openslide.OpenSlide(svs_path)
#     level_0_dims = slide.level_dimensions[0]

#     # Target dimensions (downsampled by 2 for 20x)
#     down_width = level_0_dims[0] // downsample_factor
#     down_height = level_0_dims[1] // downsample_factor

#     # Create an empty numpy array to store the full downsampled image
#     downsampled_array = np.zeros((down_height, down_width, 3), dtype=np.uint8)

#     print(f"Downsampling from {level_0_dims} to {(down_width, down_height)}")

#     for y in range(0, level_0_dims[1], tile_size):
#         for x in range(0, level_0_dims[0], tile_size):
#             # Read tile from level 0
#             read_w = min(tile_size, level_0_dims[0] - x)
#             read_h = min(tile_size, level_0_dims[1] - y)
#             tile = slide.read_region((x, y), 0, (read_w, read_h)).convert("RGB")

#             # Resize to 20x
#             tile_resized = tile.resize((read_w // downsample_factor, read_h // downsample_factor), Image.BILINEAR)
#             tile_np = np.array(tile_resized)

#             # Calculate paste coordinates
#             dx = x // downsample_factor
#             dy = y // downsample_factor
#             dw = tile_np.shape[1]
#             dh = tile_np.shape[0]

#             # Paste into main array
#             downsampled_array[dy:dy+dh, dx:dx+dw, :] = tile_np

#     print("✅ Downsampling complete. Saving as TIFF...")
#     tiff.imwrite(tiff_output_path, downsampled_array)
#     print(f"✅ Saved TIFF: {tiff_output_path}")

#     return downsampled_array


import openslide
from PIL import Image
import numpy as np
import tifffile as tiff
import os

def downsample_wsi_to_20x_tiff(svs_path, tiff_output_path, tile_size=1024, downsample_factor=2):
    slide = openslide.OpenSlide(svs_path)
    level_0_dims = slide.level_dimensions[0]

    # Target dimensions (downsampled by 2 for 20x)
    down_width = level_0_dims[0] // downsample_factor
    down_height = level_0_dims[1] // downsample_factor

    # Create an empty numpy array to store the full downsampled image
    downsampled_array = np.zeros((down_height, down_width, 3), dtype=np.uint8)

    print(f"Downsampling from {level_0_dims} to {(down_width, down_height)}")

    for y in range(0, level_0_dims[1], tile_size):
        for x in range(0, level_0_dims[0], tile_size):
            # Read tile from level 0
            read_w = min(tile_size, level_0_dims[0] - x)
            read_h = min(tile_size, level_0_dims[1] - y)
            tile = slide.read_region((x, y), 0, (read_w, read_h)).convert("RGB")

            # Resize to 20x
            tile_resized = tile.resize((read_w // downsample_factor, read_h // downsample_factor), Image.BILINEAR)
            tile_np = np.array(tile_resized)

            # Calculate paste coordinates
            dx = x // downsample_factor
            dy = y // downsample_factor
            dw = tile_np.shape[1]
            dh = tile_np.shape[0]

            # Paste into main array
            downsampled_array[dy:dy+dh, dx:dx+dw, :] = tile_np

    print("✅ Downsampling complete. Saving as TIFF...")
    tiff.imwrite(tiff_output_path, downsampled_array)
    print(f"✅ Saved TIFF: {tiff_output_path}")

    return downsampled_array


def save_preview_png(numpy_image, output_png_path, max_size=(2000, 2000)):
    # Resize image to preview size
    pil_img = Image.fromarray(numpy_image)
    pil_img.thumbnail(max_size, Image.LANCZOS)
    pil_img.save(output_png_path)
    print(f"✅ Saved preview PNG: {output_png_path}")






# # output_dir="./downsampled"
# # target_size = (1024, 1024)
# # label="msih"
# # # print(f"🚀 Processing: {svs_file} in {subdir}")
# # level= downsample_whole_wsi_to_20x(svs_path, output_dir)


# # Your WSI path
# svs_path = "D:/Aamir Gulzar/WSI_Raw_data/263abaf6-c5a1-4215-8dd5-a8b2a7bfd745/TCGA-3L-AA1B-01Z-00-DX2.17CE3683-F4B1-4978-A281-8F620C4D77B4.svs"
# tiff_output = "output_20xnew.tiff"
# png_output = "output_20x_previewnew.png"

# # Run downsampling and save preview
# wsi_array = downsample_wsi_to_20x_tiff(svs_path, tiff_output)
# save_preview_png(wsi_array, png_output)



# output_dir="./downsampled"
# target_size = (1024, 1024)
# label="msih"
# # print(f"🚀 Processing: {svs_file} in {subdir}")
# level= downsample_whole_wsi_to_20x(svs_path, output_dir)

In [ ]:
pip install tifffile


In [ ]:
# svs_file
import openslide
from PIL import Image

# Load your 40x WSI
slide = openslide.OpenSlide(svs_path)

import openslide
from PIL import Image
import numpy as np
import os


# Parameters
scale_factor = 2.0  # downsample 40x to 20x
level_0_dim = slide.level_dimensions[0]  # (143752, 57366)

# Output size at 20x
downsampled_size = (level_0_dim[0] // int(scale_factor), level_0_dim[1] // int(scale_factor))

# Tile size to read at 40x (can tune based on memory)
tile_size = 1024

# Output image canvas
downsampled_image = Image.new("RGB", downsampled_size)

# Loop through tiles
for y in range(0, level_0_dim[1], tile_size):
    for x in range(0, level_0_dim[0], tile_size):
        # Ensure we don't go out of bounds
        read_width = min(tile_size, level_0_dim[0] - x)
        read_height = min(tile_size, level_0_dim[1] - y)

        # Read patch at 40x
        patch = slide.read_region((x, y), level=0, size=(read_width, read_height)).convert("RGB")

        # Downsample patch to 20x
        patch_down = patch.resize(
            (read_width // 2, read_height // 2),
            resample=Image.BILINEAR
        )

        # Paste into final canvas
        down_x = x // 2# # Your WSI path
svs_path = "D:/Aamir Gulzar/WSI_Raw_data/263abaf6-c5a1-4215-8dd5-a8b2a7bfd745/TCGA-3L-AA1B-01Z-00-DX2.17CE3683-F4B1-4978-A281-8F620C4D77B4.svs"
tiff_output = "output_20xnew.tiff"
png_output = "output_20x_previewnew.png"

slide = openslide.OpenSlide(svs_path)
scale_factor = 2.0  # downsample 40x to 20x
level_0_dim = slide.level_dimensions[0]  # (143752, 57366)

# Output size at 20x
downsampled_size = (level_0_dim[0] // int(scale_factor), level_0_dim[1] // int(scale_factor))


# # Run downsampling and save preview
# wsi_array = downsample_wsi_to_20x_tiff(svs_path, tiff_output)
save_preview_png(wsi_array, png_output,downsampled_size)
        downsampled_image.paste(patch_down, (down_x, down_y))

# Save final image
downsampled_image.save("whole_slide_20x.png")
print("Downsampling complete: saved as 'whole_slide_20x.png'")

In [ ]:
import os
import subprocess
from pathlib import Path

# Top-level directory containing subfolders with .svs files
wsi_root = "D:/Aamir Gulzar/WSI_Raw_data"
flat_dir = "./FLAT_DIRECTORY"
count=0

# Loop through all subfolders
for subdir in os.listdir(wsi_root):
    #***************Checking for svs file***************#
    if count==2:
        break
    count+=1
    
    full_subdir_path = os.path.join(wsi_root, subdir) #D:/Aamir Gulzar/WSI_Raw_data/0ecb2d7b-0a69-46ce-80d6-1be65cec3cc7
    if not os.path.isdir(full_subdir_path):
        continue  # skip non-folder items

    # Search for .svs file in this subfolder
    svs_files = [f for f in os.listdir(full_subdir_path) if f.endswith('.svs')]
    if not svs_files:
        print(f"❌ No .svs file found in: {full_subdir_path}")
        continue
    else:
        print(svs_files)

    # If multiple .svs files exist, process each one
    for svs_file in svs_files:
        
        #***************Checking is svs is valid or not***************#
        bool_valid, label= check_svs_file(svs_file, "./slides_metadata/slide_metadata_filtered.csv", filename_column="Filename", label_column="Label")

        newnameofsvs= f"{svs_file[:12]}_{label}"
        if bool_valid==False:
            continue

        # print(bool_valid, newnameofsvs)

        #***************setting directories for wsi_raw_data***************#
        source_dir = full_subdir_path #D:/Aamir Gulzar/WSI_Raw_data/0ecb2d7b-0a69-46ce-80d6-1be65cec3cc7
        source_dir_quoted = f"{source_dir}" #D:/Aamir Gulzar/WSI_Raw_data/0ecb2d7b-0a69-46ce-80d6-1be65cec3cc7
        
        flat_dir_quoted = f"{flat_dir}" #./FLAT_DIRECTORY

        #***************setting directories for downsampled***************#
        output_dir= "./FLAT_DIRECTORY/downsampled"
        
        svs_path = os.path.join(source_dir_quoted, svs_file) #D:/Aamir Gulzar/WSI_Raw_data/0ecb2d7b-0a69-46ce-80d6-1be65cec3cc7/TCGA-AA-3524-01Z-00-DX1.b1aae264-87be-4514-8f9d-25660b39caa7.svs
        print(f"🚀 Processing: {svs_file} in {subdir}")

        # slide = openslide.OpenSlide(svs_path)
        # scale_factor = 2.0  # downsample 40x to 20x
        # level_0_dim = slide.level_dimensions[0]  # (143752, 57366)
        
        # # Output size at 20x
        # downsampled_size = (level_0_dim[0] // int(scale_factor), level_0_dim[1] // int(scale_factor))
        
        # # # Run downsampling and save preview
        # wsi_array, subfolder_path = downsample_wsi_to_20x_tiff(svs_path,label) #D:/Aamir Gulzar/KSA_project2/data_preprocessing/TIFF/TCGA-A6-2686_MSIH
        # output_dir = Path("D:/Aamir Gulzar/KSA_project2/data_preprocessing/TIFF")
        
        # # output_path= os.path.join(output_dir, tiff_filename)
        # save_preview_png(wsi_array, svs_path,label,downsampled_size)

        # #***************Creating .h5 command***************#
        # command = [
        #     'python', './CLAM/create_patches_fp.py',
        #     '--source', subfolder_path,
        #     '--save_dir', flat_dir,
        #     '--patch_size', '512',
        #     '--step_size','512',
        #     '--seg', '--patch'
        # ]

        # # Execute the command
        # result = subprocess.run(command, capture_output=True, text=True)

        # if result.returncode == 0:
        #     print(f"✅ Success: {svs_file}")
            
        #     #***************Renaming .h5 file***************#
        #     new_h5_path= rename_h5_file(svs_file, label, h5_dir="./FLAT_DIRECTORY/patches")
        # else:
        #     print(f"❌ Error processing {svs_file}:\n{result.stderr}")
        #     continue

        # #***************Setting path for png patches***************#
        # base_dir = "./FLAT_DIRECTORY"
        # sub_dir = "patches"

        # # Create full path
        # print(new_h5_path)
        # base_dir= "./Patches"
        # sub_dir= f"{svs_file[:12]}_{label}"
        # output_dir = os.path.join(base_dir, sub_dir)

        # df = pd.read_csv(csv_path)
        
        # # Create a new row (dict)
        # new_row = {
        #     'svs_file': svs_file,
        #     'renamed_file': sub_dir
        # }
        
        # # Append the new row
        # df = pd.concat([df, pd.DataFrame([new_row])], ignore_index=True)
        
        # # Save back to CSV
        # df.to_csv(csv_path, index=False)
        # print(output_dir)

        # #creating patches from h5 files
        # create_patches_from_h5(new_h5_path, svs_path, output_dir)

        

In [ ]:
# internal imports
from wsi_core.WholeSlideImage import WholeSlideImage
from wsi_core.wsi_utils import StitchCoords
from wsi_core.batch_process_utils import initialize_df
# other imports
import os
import numpy as np
import time
import argparse
import pdb
import pandas as pd
from tqdm import tqdm

OPENSLIDE_PATH = r'D:\Aamir Gulzar\openslide-bin-4.0.0.3-windows-x64\bin'
if hasattr(os, 'add_dll_directory'):
    # Windows
    with os.add_dll_directory(OPENSLIDE_PATH):
        import openslide
else:
    import openslide

def stitching(file_path, wsi_object, downscale = 64):
	start = time.time()
	heatmap = StitchCoords(file_path, wsi_object, downscale=downscale, bg_color=(0,0,0), alpha=-1, draw_grid=False)
	total_time = time.time() - start
	
	return heatmap, total_time

def segment(WSI_object, seg_params = None, filter_params = None, mask_file = None):
	### Start Seg Timer
	start_time = time.time()
	# Use segmentation file
	if mask_file is not None:
		WSI_object.initSegmentation(mask_file)
	# Segment	
	else:
		WSI_object.segmentTissue(**seg_params, filter_params=filter_params)

	### Stop Seg Timers
	seg_time_elapsed = time.time() - start_time   
	return WSI_object, seg_time_elapsed

def patching(WSI_object, **kwargs):
	### Start Patch Timer
	start_time = time.time()

	# Patch
	file_path = WSI_object.process_contours(**kwargs)


	### Stop Patch Timer
	patch_time_elapsed = time.time() - start_time
	return file_path, patch_time_elapsed


def seg_and_patch(source, save_dir, patch_save_dir, mask_save_dir, stitch_save_dir, 
				  patch_size = 256, step_size = 256, 
				  seg_params = {'seg_level': -1, 'sthresh': 8, 'mthresh': 7, 'close': 4, 'use_otsu': False,
				  'keep_ids': 'none', 'exclude_ids': 'none'},
				  filter_params = {'a_t':100, 'a_h': 16, 'max_n_holes':8}, 
				  vis_params = {'vis_level': -1, 'line_thickness': 500},
				  patch_params = {'use_padding': True, 'contour_fn': 'four_pt'},
				  patch_level = 0,
				  use_default_params = False, 
				  seg = False, save_mask = False, 
				  stitch= False, 
				  patch = False, auto_skip=True, process_list = None):
	


	slides = sorted(os.listdir(source))
	slides = [slide for slide in slides if os.path.isfile(os.path.join(source, slide))]
	if process_list is None:
		df = initialize_df(slides, seg_params, filter_params, vis_params, patch_params)
	
	else:
		df = pd.read_csv(process_list)
		df = initialize_df(df, seg_params, filter_params, vis_params, patch_params)

	mask = df['process'] == 1
	process_stack = df[mask]

	total = len(process_stack)

	legacy_support = 'a' in df.keys()
	if legacy_support:
		print('detected legacy segmentation csv file, legacy support enabled')
		df = df.assign(**{'a_t': np.full((len(df)), int(filter_params['a_t']), dtype=np.uint32),
		'a_h': np.full((len(df)), int(filter_params['a_h']), dtype=np.uint32),
		'max_n_holes': np.full((len(df)), int(filter_params['max_n_holes']), dtype=np.uint32),
		'line_thickness': np.full((len(df)), int(vis_params['line_thickness']), dtype=np.uint32),
		'contour_fn': np.full((len(df)), patch_params['contour_fn'])})

	seg_times = 0.
	patch_times = 0.
	stitch_times = 0.

	for i in tqdm(range(total)):
		df.to_csv(os.path.join(save_dir, 'process_list_autogen.csv'), index=False)
		idx = process_stack.index[i]
		slide = process_stack.loc[idx, 'slide_id']
		print("\n\nprogress: {:.2f}, {}/{}".format(i/total, i, total))
		print('processing {}'.format(slide))
		
		df.loc[idx, 'process'] = 0
		slide_id, _ = os.path.splitext(slide)
		# slide_id = slide_id[:12]

		if auto_skip and os.path.isfile(os.path.join(patch_save_dir, slide_id + '.h5')):
			print('{} already exist in destination location, skipped'.format(slide_id))
			df.loc[idx, 'status'] = 'already_exist'
			continue

		# Inialize WSI
		full_path = os.path.join(source, slide)
		WSI_object = WholeSlideImage(full_path)

		if use_default_params:
			current_vis_params = vis_params.copy()
			current_filter_params = filter_params.copy()
			current_seg_params = seg_params.copy()
			current_patch_params = patch_params.copy()
			
		else:
			current_vis_params = {}
			current_filter_params = {}
			current_seg_params = {}
			current_patch_params = {}


			for key in vis_params.keys():
				if legacy_support and key == 'vis_level':
					df.loc[idx, key] = -1
				current_vis_params.update({key: df.loc[idx, key]})

			for key in filter_params.keys():
				if legacy_support and key == 'a_t':
					old_area = df.loc[idx, 'a']
					seg_level = df.loc[idx, 'seg_level']
					scale = WSI_object.level_downsamples[seg_level]
					adjusted_area = int(old_area * (scale[0] * scale[1]) / (512 * 512))
					current_filter_params.update({key: adjusted_area})
					df.loc[idx, key] = adjusted_area
				current_filter_params.update({key: df.loc[idx, key]})

			for key in seg_params.keys():
				if legacy_support and key == 'seg_level':
					df.loc[idx, key] = -1
				current_seg_params.update({key: df.loc[idx, key]})

			for key in patch_params.keys():
				current_patch_params.update({key: df.loc[idx, key]})

		if current_vis_params['vis_level'] < 0:
			if len(WSI_object.level_dim) == 1:
				current_vis_params['vis_level'] = 0
			
			else:	
				wsi = WSI_object.getOpenSlide()
				best_level = wsi.get_best_level_for_downsample(64)
				current_vis_params['vis_level'] = best_level

		if current_seg_params['seg_level'] < 0:
			if len(WSI_object.level_dim) == 1:
				current_seg_params['seg_level'] = 0
			
			else:
				wsi = WSI_object.getOpenSlide()
				best_level = wsi.get_best_level_for_downsample(64)
				current_seg_params['seg_level'] = best_level

		keep_ids = str(current_seg_params['keep_ids'])
		if keep_ids != 'none' and len(keep_ids) > 0:
			str_ids = current_seg_params['keep_ids']
			current_seg_params['keep_ids'] = np.array(str_ids.split(',')).astype(int)
		else:
			current_seg_params['keep_ids'] = []

		exclude_ids = str(current_seg_params['exclude_ids'])
		if exclude_ids != 'none' and len(exclude_ids) > 0:
			str_ids = current_seg_params['exclude_ids']
			current_seg_params['exclude_ids'] = np.array(str_ids.split(',')).astype(int)
		else:
			current_seg_params['exclude_ids'] = []

		w, h = WSI_object.level_dim[current_seg_params['seg_level']] 
		if w * h > 1e8:
			print('level_dim {} x {} is likely too large for successful segmentation, aborting'.format(w, h))
			df.loc[idx, 'status'] = 'failed_seg'
			continue

		df.loc[idx, 'vis_level'] = current_vis_params['vis_level']
		df.loc[idx, 'seg_level'] = current_seg_params['seg_level']


		seg_time_elapsed = -1
		if seg:
			WSI_object, seg_time_elapsed = segment(WSI_object, current_seg_params, current_filter_params) 

		if save_mask:
			mask = WSI_object.visWSI(**current_vis_params)
			mask_path = os.path.join(mask_save_dir, slide_id+'.jpg')
			mask.save(mask_path)

		patch_time_elapsed = -1 # Default time
		if patch:
			current_patch_params.update({'patch_level': patch_level, 'patch_size': patch_size, 'step_size': step_size, 
										 'save_path': patch_save_dir})
			file_path, patch_time_elapsed = patching(WSI_object = WSI_object,  **current_patch_params,)
		
		stitch_time_elapsed = -1
		if stitch:
			file_path = os.path.join(patch_save_dir, slide_id+'.h5')
			if os.path.isfile(file_path):
				heatmap, stitch_time_elapsed = stitching(file_path, WSI_object, downscale=64)
				stitch_path = os.path.join(stitch_save_dir, slide_id+'.jpg')
				heatmap.save(stitch_path)

		print("segmentation took {} seconds".format(seg_time_elapsed))
		print("patching took {} seconds".format(patch_time_elapsed))
		print("stitching took {} seconds".format(stitch_time_elapsed))
		df.loc[idx, 'status'] = 'processed'

		seg_times += seg_time_elapsed
		patch_times += patch_time_elapsed
		stitch_times += stitch_time_elapsed

	seg_times /= total
	patch_times /= total
	stitch_times /= total

	df.to_csv(os.path.join(save_dir, 'process_list_autogen.csv'), index=False)
	print("average segmentation time in s per slide: {}".format(seg_times))
	print("average patching time in s per slide: {}".format(patch_times))
	print("average stiching time in s per slide: {}".format(stitch_times))
		
	return seg_times, patch_times

parser = argparse.ArgumentParser(description='seg and patch')
parser.add_argument('--source', type = str,
					help='path to folder containing raw wsi image files')
parser.add_argument('--step_size', type = int, default=256,
					help='step_size')
parser.add_argument('--patch_size', type = int, default=256,
					help='patch_size')
parser.add_argument('--patch', default=False, action='store_true')
parser.add_argument('--seg', default=False, action='store_true')
parser.add_argument('--stitch', default=False, action='store_true')
parser.add_argument('--no_auto_skip', default=True, action='store_false')
parser.add_argument('--save_dir', type = str,
					help='directory to save processed data')
parser.add_argument('--preset', default=None, type=str,
					help='predefined profile of default segmentation and filter parameters (.csv)')
parser.add_argument('--patch_level', type=int, default=0, 
					help='downsample level at which to patch')
parser.add_argument('--process_list',  type = str, default=None,
					help='name of list of images to process with parameters (.csv)')

if __name__ == '__main__':
	args = parser.parse_args()

	patch_save_dir = os.path.join(args.save_dir, 'patches')
	mask_save_dir = os.path.join(args.save_dir, 'masks')
	stitch_save_dir = os.path.join(args.save_dir, 'stitches')

	if args.process_list:
		process_list = os.path.join(args.save_dir, args.process_list)

	else:
		process_list = None

	print('source: ', args.source)
	print('patch_save_dir: ', patch_save_dir)
	print('mask_save_dir: ', mask_save_dir)
	print('stitch_save_dir: ', stitch_save_dir)
	
	directories = {'source': args.source, 
				   'save_dir': args.save_dir,
				   'patch_save_dir': patch_save_dir, 
				   'mask_save_dir' : mask_save_dir, 
				   'stitch_save_dir': stitch_save_dir} 

	for key, val in directories.items():
		print("{} : {}".format(key, val))
		if key not in ['source']:
			os.makedirs(val, exist_ok=True)

	seg_params = {'seg_level': -1, 'sthresh': 8, 'mthresh': 7, 'close': 4, 'use_otsu': False,
				  'keep_ids': 'none', 'exclude_ids': 'none'}
	filter_params = {'a_t':100, 'a_h': 16, 'max_n_holes':8}
	vis_params = {'vis_level': -1, 'line_thickness': 250}
	patch_params = {'use_padding': True, 'contour_fn': 'four_pt'}

	if args.preset:
		preset_df = pd.read_csv(os.path.join('presets', args.preset))
		for key in seg_params.keys():
			seg_params[key] = preset_df.loc[0, key]

		for key in filter_params.keys():
			filter_params[key] = preset_df.loc[0, key]

		for key in vis_params.keys():
			vis_params[key] = preset_df.loc[0, key]

		for key in patch_params.keys():
			patch_params[key] = preset_df.loc[0, key]
	
	parameters = {'seg_params': seg_params,
				  'filter_params': filter_params,
	 			  'patch_params': patch_params,
				  'vis_params': vis_params}

	print(parameters)
	slide_id = os.path.basename(args.source)

	row_data = {}
	row_data = {'slide_id': slide_id}

	for param_group in parameters.values():
		row_data.update(param_group)

	csv_path = r"D:\Aamir Gulzar\KSA_project2\data_preprocessing\FLAT_DIRECTORY\process_list_final.csv"
	df = pd.read_csv(csv_path)

	# Append new row
	df = pd.concat([df, pd.DataFrame([row_data])], ignore_index=True)

	# Save back to CSV
	df.to_csv(csv_path, index=False)

	try:
		# Pick a representative SVS file from the source folder
		sample_files = [f for f in os.listdir(args.source) if f.endswith('.svs')]
		if not sample_files:
			raise FileNotFoundError("No .svs files found in source directory.")

		sample_slide_path = os.path.join(args.source, sample_files[0])
		slide = openslide.OpenSlide(sample_slide_path)

		# Get WSI magnification and compute scale factor
		mag = int(slide.properties.get("openslide.objective-power", 20))  # fallback to 20x
		scale_factor = mag / 20  # e.g., 40 / 20 = 2
		patch_level = slide.get_best_level_for_downsample(scale_factor)

		print(f"Computed patch_level = {patch_level} for {mag}x WSI (target 20x)")
		args.patch_level = patch_level  # override the patch_level from CLI
	except Exception as e:
		print(f"Failed to determine patch_level automatically: {e}")
		print(f"Using default patch_level = {args.patch_level}")

	seg_times, patch_times = seg_and_patch(**directories, **parameters,
											patch_size = args.patch_size, step_size=args.step_size, 
											seg = args.seg,  use_default_params=False, save_mask = False, 
											stitch= args.stitch,
											patch_level=args.patch_level, patch = args.patch,
											process_list = process_list, auto_skip=args.no_auto_skip)

In [ ]:
# internal imports
from wsi_core.WholeSlideImage import WholeSlideImage
from wsi_core.wsi_utils import StitchCoords
from wsi_core.batch_process_utils import initialize_df
# other imports
import os
import numpy as np
import time
import argparse
import pdb
import pandas as pd
from tqdm import tqdm

OPENSLIDE_PATH = r'D:\Aamir Gulzar\openslide-bin-4.0.0.3-windows-x64\bin'
if hasattr(os, 'add_dll_directory'):
    # Windows
    with os.add_dll_directory(OPENSLIDE_PATH):
        import openslide
else:
    import openslide

def stitching(file_path, wsi_object, downscale = 64):
    start = time.time()
    heatmap = StitchCoords(file_path, wsi_object, downscale=downscale, bg_color=(0,0,0), alpha=-1, draw_grid=False)
    total_time = time.time() - start
    
    return heatmap, total_time

def segment(WSI_object, seg_params = None, filter_params = None, mask_file = None):
    ### Start Seg Timer
    start_time = time.time()
    # Use segmentation file
    if mask_file is not None:
        WSI_object.initSegmentation(mask_file)
    # Segment   
    else:
        WSI_object.segmentTissue(**seg_params, filter_params=filter_params)

    ### Stop Seg Timers
    seg_time_elapsed = time.time() - start_time   
    return WSI_object, seg_time_elapsed

def patching(WSI_object, **kwargs):
    ### Start Patch Timer
    start_time = time.time()

    # Patch
    file_path = WSI_object.process_contours(**kwargs)


    ### Stop Patch Timer
    patch_time_elapsed = time.time() - start_time
    return file_path, patch_time_elapsed


def seg_and_patch(source, save_dir, patch_save_dir, mask_save_dir, stitch_save_dir, 
                  patch_size = 512, step_size = 512, 
                  seg_params = {'seg_level': -1, 'sthresh': 8, 'mthresh': 7, 'close': 4, 'use_otsu': False,
                  'keep_ids': 'none', 'exclude_ids': 'none'},
                  filter_params = {'a_t':100, 'a_h': 16, 'max_n_holes':8}, 
                  vis_params = {'vis_level': -1, 'line_thickness': 500},
                  patch_params = {'use_padding': True, 'contour_fn': 'four_pt'},
                  patch_level = 0,
                  use_default_params = False, 
                  seg = False, save_mask = False, 
                  stitch= False, 
                  patch = False, auto_skip=True, process_list = None):
    


    slides = sorted(os.listdir(source))
    slides = [slide for slide in slides if os.path.isfile(os.path.join(source, slide))]
    if process_list is None:
        df = initialize_df(slides, seg_params, filter_params, vis_params, patch_params)
    
    else:
        df = pd.read_csv(process_list)
        df = initialize_df(df, seg_params, filter_params, vis_params, patch_params)

    mask = df['process'] == 1
    process_stack = df[mask]

    total = len(process_stack)

    legacy_support = 'a' in df.keys()
    if legacy_support:
        print('detected legacy segmentation csv file, legacy support enabled')
        df = df.assign(**{'a_t': np.full((len(df)), int(filter_params['a_t']), dtype=np.uint32),
        'a_h': np.full((len(df)), int(filter_params['a_h']), dtype=np.uint32),
        'max_n_holes': np.full((len(df)), int(filter_params['max_n_holes']), dtype=np.uint32),
        'line_thickness': np.full((len(df)), int(vis_params['line_thickness']), dtype=np.uint32),
        'contour_fn': np.full((len(df)), patch_params['contour_fn'])})

    seg_times = 0.
    patch_times = 0.
    stitch_times = 0.

    for i in tqdm(range(total)):
        df.to_csv(os.path.join(save_dir, 'process_list_autogen.csv'), index=False)
        idx = process_stack.index[i]
        slide = process_stack.loc[idx, 'slide_id']
        print("\n\nprogress: {:.2f}, {}/{}".format(i/total, i, total))
        print('processing {}'.format(slide))
        
        df.loc[idx, 'process'] = 0
        slide_id, _ = os.path.splitext(slide)
        # slide_id = slide_id[:12]

        if auto_skip and os.path.isfile(os.path.join(patch_save_dir, slide_id + '.h5')):
            print('{} already exist in destination location, skipped'.format(slide_id))
            df.loc[idx, 'status'] = 'already_exist'
            continue

        # Inialize WSI
        full_path = os.path.join(source, slide)
        WSI_object = WholeSlideImage(full_path)

        if use_default_params:
            current_vis_params = vis_params.copy()
            current_filter_params = filter_params.copy()
            current_seg_params = seg_params.copy()
            current_patch_params = patch_params.copy()
            
        else:
            current_vis_params = {}
            current_filter_params = {}
            current_seg_params = {}
            current_patch_params = {}


            for key in vis_params.keys():
                if legacy_support and key == 'vis_level':
                    df.loc[idx, key] = -1
                current_vis_params.update({key: df.loc[idx, key]})

            for key in filter_params.keys():
                if legacy_support and key == 'a_t':
                    old_area = df.loc[idx, 'a']
                    seg_level = df.loc[idx, 'seg_level']
                    scale = WSI_object.level_downsamples[seg_level]
                    adjusted_area = int(old_area * (scale[0] * scale[1]) / (512 * 512))
                    current_filter_params.update({key: adjusted_area})
                    df.loc[idx, key] = adjusted_area
                current_filter_params.update({key: df.loc[idx, key]})

            for key in seg_params.keys():
                if legacy_support and key == 'seg_level':
                    df.loc[idx, key] = -1
                current_seg_params.update({key: df.loc[idx, key]})

            for key in patch_params.keys():
                current_patch_params.update({key: df.loc[idx, key]})

        if current_vis_params['vis_level'] < 0:
            if len(WSI_object.level_dim) == 1:
                current_vis_params['vis_level'] = 0
            
            else:   
                wsi = WSI_object.getOpenSlide()
                best_level = wsi.get_best_level_for_downsample(64)
                current_vis_params['vis_level'] = best_level

        if current_seg_params['seg_level'] < 0:
            if len(WSI_object.level_dim) == 1:
                current_seg_params['seg_level'] = 0
            
            else:
                wsi = WSI_object.getOpenSlide()
                best_level = wsi.get_best_level_for_downsample(64)
                current_seg_params['seg_level'] = best_level

        keep_ids = str(current_seg_params['keep_ids'])
        if keep_ids != 'none' and len(keep_ids) > 0:
            str_ids = current_seg_params['keep_ids']
            current_seg_params['keep_ids'] = np.array(str_ids.split(',')).astype(int)
        else:
            current_seg_params['keep_ids'] = []

        exclude_ids = str(current_seg_params['exclude_ids'])
        if exclude_ids != 'none' and len(exclude_ids) > 0:
            str_ids = current_seg_params['exclude_ids']
            current_seg_params['exclude_ids'] = np.array(str_ids.split(',')).astype(int)
        else:
            current_seg_params['exclude_ids'] = []

        w, h = WSI_object.level_dim[current_seg_params['seg_level']] 
        if w * h > 10e8:
            print('level_dim {} x {} is likely too large for successful segmentation, aborting'.format(w, h))
            df.loc[idx, 'status'] = 'failed_seg'
            continue

        df.loc[idx, 'vis_level'] = current_vis_params['vis_level']
        df.loc[idx, 'seg_level'] = current_seg_params['seg_level']


        seg_time_elapsed = -1
        if seg:
            WSI_object, seg_time_elapsed = segment(WSI_object, current_seg_params, current_filter_params)

        if save_mask:
            mask = WSI_object.visWSI(**current_vis_params)
            mask_path = os.path.join(mask_save_dir, slide_id+'.jpg')
            mask.save(mask_path)

        patch_time_elapsed = -1 # Default time
        if patch:
            current_patch_params.update({'patch_level': patch_level, 'patch_size': patch_size, 'step_size': step_size, 
                                         'save_path': patch_save_dir})
            file_path, patch_time_elapsed = patching(WSI_object = WSI_object,  **current_patch_params,)
        
        stitch_time_elapsed = -1
        if stitch:
            file_path = os.path.join(patch_save_dir, slide_id+'.h5')
            if os.path.isfile(file_path):
                heatmap, stitch_time_elapsed = stitching(file_path, WSI_object, downscale=64)
                stitch_path = os.path.join(stitch_save_dir, slide_id+'.jpg')
                heatmap.save(stitch_path)

        print("segmentation took {} seconds".format(seg_time_elapsed))
        print("patching took {} seconds".format(patch_time_elapsed))
        print("stitching took {} seconds".format(stitch_time_elapsed))
        df.loc[idx, 'status'] = 'processed'

        seg_times += seg_time_elapsed
        patch_times += patch_time_elapsed
        stitch_times += stitch_time_elapsed

    seg_times /= total
    patch_times /= total
    stitch_times /= total

    df.to_csv(os.path.join(save_dir, 'process_list_autogen.csv'), index=False)
    print("average segmentation time in s per slide: {}".format(seg_times))
    print("average patching time in s per slide: {}".format(patch_times))
    print("average stiching time in s per slide: {}".format(stitch_times))
        
    return seg_times, patch_times

parser = argparse.ArgumentParser(description='seg and patch')
parser.add_argument('--source', type = str,
                    help='path to folder containing raw wsi image files')
parser.add_argument('--step_size', type = int, default=256,
                    help='step_size')
parser.add_argument('--patch_size', type = int, default=256,
                    help='patch_size')
parser.add_argument('--patch', default=False, action='store_true')
parser.add_argument('--seg', default=False, action='store_true')
parser.add_argument('--stitch', default=False, action='store_true')
parser.add_argument('--no_auto_skip', default=True, action='store_false')
parser.add_argument('--save_dir', type = str,
                    help='directory to save processed data')
parser.add_argument('--preset', default=None, type=str,
                    help='predefined profile of default segmentation and filter parameters (.csv)')
parser.add_argument('--patch_level', type=int, default=0, 
                    help='downsample level at which to patch')
parser.add_argument('--process_list',  type = str, default=None,
                    help='name of list of images to process with parameters (.csv)')

if __name__ == '__main__':
    args = parser.parse_args()

    patch_save_dir = os.path.join(args.save_dir, 'patches')
    mask_save_dir = os.path.join(args.save_dir, 'masks')
    stitch_save_dir = os.path.join(args.save_dir, 'stitches')

    if args.process_list:
        process_list = os.path.join(args.save_dir, args.process_list)

    else:
        process_list = None

    print('source: ', args.source)
    print('patch_save_dir: ', patch_save_dir)
    print('mask_save_dir: ', mask_save_dir)
    print('stitch_save_dir: ', stitch_save_dir)
    
    directories = {'source': args.source, 
                   'save_dir': args.save_dir,
                   'patch_save_dir': patch_save_dir, 
                   'mask_save_dir' : mask_save_dir, 
                   'stitch_save_dir': stitch_save_dir} 

    for key, val in directories.items():
        print("{} : {}".format(key, val))
        if key not in ['source']:
            os.makedirs(val, exist_ok=True)

    seg_params = {'seg_level': -1, 'sthresh': 8, 'mthresh': 7, 'close': 4, 'use_otsu': False,
                  'keep_ids': 'none', 'exclude_ids': 'none'}
    filter_params = {'a_t':100, 'a_h': 16, 'max_n_holes':8}
    vis_params = {'vis_level': -1, 'line_thickness': 250}
    patch_params = {'use_padding': True, 'contour_fn': 'four_pt'}

    if args.preset:
        preset_df = pd.read_csv(os.path.join('presets', args.preset))
        for key in seg_params.keys():
            seg_params[key] = preset_df.loc[0, key]

        for key in filter_params.keys():
            filter_params[key] = preset_df.loc[0, key]

        for key in vis_params.keys():
            vis_params[key] = preset_df.loc[0, key]

        for key in patch_params.keys():
            patch_params[key] = preset_df.loc[0, key]
    
    parameters = {'seg_params': seg_params,
                  'filter_params': filter_params,
                  'patch_params': patch_params,
                  'vis_params': vis_params}

    print(parameters)

    slide_id = os.path.basename(args.source)
    row_data = {}
    row_data = {'slide_id': slide_id}

    for param_group in parameters.values():
        row_data.update(param_group)

    # csv_path = r"D:\Aamir Gulzar\KSA_project2\data_preprocessing\Cancer-detection-classifier\FLAT_DIRECTORY\process_list_final.csv"
    # df = pd.read_csv(csv_path)

    # # Append new row
    # df = pd.concat([df, pd.DataFrame([row_data])], ignore_index=True)

    # # Save back to CSV
    # df.to_csv(csv_path, index=False)

    # try:
    #     # Pick a representative SVS file from the source folder
    #     sample_files = [f for f in os.listdir(args.source) if f.endswith('.svs')]
    #     if not sample_files:
    #         raise FileNotFoundError("No .svs files found in source directory.")

    #     sample_slide_path = os.path.join(args.source, sample_files[0])
    #     slide = openslide.OpenSlide(sample_slide_path)

    #     # Get WSI magnification and compute scale factor
    #     mag = int(slide.properties.get("openslide.objective-power", 20))  # fallback to 20x
    #     scale_factor = mag / 20  # e.g., 40 / 20 = 2
    #     patch_level = slide.get_best_level_for_downsample(scale_factor)

    #     print(f"Computed patch_level = {patch_level} for {mag}x WSI (target 20x)")
    #     args.patch_level = patch_level  # override the patch_level from CLI
    # except Exception as e:
    #     print(f"Failed to determine patch_level automatically: {e}")
    #     print(f"Using default patch_level = {args.patch_level}")

    seg_times, patch_times = seg_and_patch(**directories, **parameters,
                                            patch_size = args.patch_size, step_size=args.step_size, 
                                            seg = args.seg,  use_default_params=False, save_mask = False, 
                                            stitch= args.stitch,
                                            patch_level=args.patch_level, patch = args.patch,
                                            process_list = process_list, auto_skip=args.no_auto_skip)